# Region Resolved Machine Learning of XANES

This notebook is intended for the public GitHub repository. It documents the executable workflow from raw simulated XAS spectra to the processed tables and analysis figures used in the manuscript.

It addresses three reviewer requests:

1. Clear run instructions, expected outputs, and dataset layout.
2. An explicit preprocessing pipeline for Gaussian smoothing, area normalization, and processed CSV generation.
3. A data dictionary and named feature construction, avoiding hard-coded positional column slicing.


## Expected repository layout

Recommended public layout:

```text
Kotsugi_paper/
  Region-Resolved-Machine-Learning-of-XANES.ipynb
  data.csv                         # processed per-spectrum table, optional if raw data are released
  data_interpolated.csv            # processed common-grid table used by ML/PCA scripts
  raw_xas/                         # optional raw simulated .dat spectra, two columns: energy intensity
  processed_xas/smoothed/          # generated Gaussian-smoothed spectra
  processed_xas/normalized/        # generated area-normalized spectra
  dimensionality_reduction_265_290/
```

If raw spectra cannot be released, keep the processed data files and retain Sections 2-4 as quantitative documentation of the preprocessing steps used to generate them.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

from scipy.ndimage import gaussian_filter1d
try:
    from scipy.integrate import simpson as _simpson
    def integrate_area(y, x):
        return _simpson(y, x=x)
except ImportError:
    from scipy.integrate import simps as _simps
    def integrate_area(y, x):
        return _simps(y, x)

ROOT = Path.cwd()
if not (ROOT / "data_interpolated.csv").exists() and (ROOT / "Kotsugi_paper").exists():
    ROOT = ROOT / "Kotsugi_paper"

RAW_XAS_DIR = ROOT / "raw_xas"
SMOOTHED_DIR = ROOT / "processed_xas" / "smoothed"
NORMALIZED_DIR = ROOT / "processed_xas" / "normalized"
DATA_CSV = ROOT / "data.csv"
DATA_INTERPOLATED_CSV = ROOT / "data_interpolated.csv"
OUT_DIR = ROOT / "dimensionality_reduction_265_290"

RANDOM_STATE = 42
ENERGY_WINDOWS = {
    "PI": (265.0, 273.5),
    "SIGMA": (273.5, 283.5),
    "POSTEDGE": (283.5, 290.0),
    "FULL": (265.0, 290.0),
}

CONCENTRATION_BINS = [0, 1.39, 2.78, 4.17, 5.56, 6.95, np.inf]
CONCENTRATION_LABELS = ["0%", "1.39%", "2.78%", "4.17%", "5.56%", "6.95%"]
TYPE_LABELS = {0: "pristine", 1: "BG", 2: "NG"}

print(f"Project root: {ROOT}")


## 1. Raw XAS file convention

Each raw simulated XAS file is expected to be a whitespace-separated `.dat` file with at least two columns:

| Column | Meaning |
|---|---|
| 0 | Photon energy in eV |
| 1 | XAS intensity |

Any additional columns are ignored by this public preprocessing workflow.


In [ ]:
def read_spectrum_dat(path):
    """Read a whitespace-separated spectrum and keep the first two columns."""
    data = pd.read_csv(path, sep=r"\s+", header=None, comment="#")
    if data.shape[1] < 2:
        raise ValueError(f"{path} has fewer than two columns")
    out = data.iloc[:, :2].copy()
    out.columns = ["Energy", "Intensity"]
    out["Energy"] = pd.to_numeric(out["Energy"], errors="raise")
    out["Intensity"] = pd.to_numeric(out["Intensity"], errors="raise")
    return out


## 2. Gaussian smoothing

The historical preprocessing used a one-dimensional Gaussian filter on intensity values only:

- `scipy.ndimage.gaussian_filter1d`
- `sigma = 1` grid point
- energy values are not changed
- output format remains two whitespace-separated columns without header


In [ ]:
def gaussian_smooth_directory(source_dir, target_dir, sigma=1.0):
    source_dir = Path(source_dir)
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    records = []
    for path in sorted(source_dir.glob("*.dat")):
        spec = read_spectrum_dat(path)
        smoothed = spec.copy()
        smoothed["Intensity"] = gaussian_filter1d(spec["Intensity"].to_numpy(), sigma=sigma)
        out_path = target_dir / path.name
        smoothed.to_csv(out_path, sep=" ", index=False, header=False)
        records.append({"file_name": path.name, "input": str(path), "output": str(out_path), "sigma": sigma})

    return pd.DataFrame(records)

# Example run when raw spectra are available:
# smoothing_log = gaussian_smooth_directory(RAW_XAS_DIR, SMOOTHED_DIR, sigma=1.0)
# smoothing_log.head()


## 3. Area normalization

The smoothed spectrum is normalized by its integrated area:

`normalized_intensity(E) = smoothed_intensity(E) / integral smoothed_intensity(E) dE`

The integral is computed using Simpson integration over the available energy grid. Spectra with all-zero intensity or zero integrated area are skipped and reported.


In [ ]:
def normalize_spectrum_file(file_path):
    spec = read_spectrum_dat(file_path)
    intensity_sum = spec["Intensity"].sum()
    if intensity_sum == 0:
        return None, "all_zero_intensity"

    area = integrate_area(spec["Intensity"].to_numpy(), spec["Energy"].to_numpy())
    if area == 0 or not np.isfinite(area):
        return None, "zero_or_invalid_area"

    normalized = spec.copy()
    normalized["Intensity"] = normalized["Intensity"] / area
    return normalized, "ok"


def normalize_directory(input_dir, output_dir):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    records = []
    for path in sorted(input_dir.glob("*.dat")):
        normalized, status = normalize_spectrum_file(path)
        out_path = output_dir / path.name
        if normalized is not None:
            normalized.to_csv(out_path, sep=" ", index=False, header=False)
        records.append({"file_name": path.name, "status": status, "output": str(out_path) if normalized is not None else ""})

    return pd.DataFrame(records)

# Example run after smoothing:
# normalization_log = normalize_directory(SMOOTHED_DIR, NORMALIZED_DIR)
# normalization_log["status"].value_counts()


## 4. Generate `data.csv` from normalized spectra

`data.csv` is a per-spectrum wide table. Each row is one spectrum. The first columns are paired energy/intensity columns (`x1`, `y1`, `x2`, `y2`, ...), followed by metadata and target columns.

If a metadata table is available, it should contain a `file_name` column and any target/label columns to merge. If no metadata table is provided, this function still creates the spectral part.


In [1]:
def build_data_csv(normalized_dir, output_csv, metadata_csv=None, max_points=1000):
    normalized_dir = Path(normalized_dir)
    records = []

    for path in sorted(normalized_dir.glob("*.dat")):
        spec = read_spectrum_dat(path)
        row = {"file_name": path.name}
        for i, (_, r) in enumerate(spec.iloc[:max_points].iterrows(), start=1):
            row[f"x{i}"] = float(r["Energy"])
            row[f"y{i}"] = float(r["Intensity"])
        records.append(row)

    data = pd.DataFrame(records)

    if metadata_csv is not None:
        metadata = pd.read_csv(metadata_csv)
        if "file_name" not in metadata.columns:
            raise ValueError("metadata_csv must contain a file_name column")
        data = data.merge(metadata, on="file_name", how="left", validate="one_to_one")

    data.to_csv(output_csv, index=False)
    return data

# Example run when normalized spectra and metadata are available:
# processed = build_data_csv(NORMALIZED_DIR, DATA_CSV, metadata_csv=ROOT / "metadata.csv")
# processed.shape


## 5. Interpolate to a common energy grid

`data_interpolated.csv` uses a different layout from `data.csv`:

- rows with numeric index values are photon energies
- each spectrum is one column
- metadata rows are appended after the energy rows
- the common grid is the intersection of all available spectral energy ranges
- the current interpolation step is 0.1 eV


In [4]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

# Bootstrap paths so this cell can be run directly from Section 5.
ROOT = Path.cwd()
if not (ROOT / "data.csv").exists() and (ROOT / "Kotsugi_paper").exists():
    ROOT = ROOT / "Kotsugi_paper"
DATA_CSV = ROOT / "data.csv"
DATA_INTERPOLATED_CSV = ROOT / "data_interpolated.csv"


def _dedupe_names(names):
    """Disambiguate repeated file names (e.g. replicate spectra) so each becomes its own column."""
    seen = {}
    deduped = []
    for name in names:
        seen[name] = seen.get(name, 0) + 1
        deduped.append(name if seen[name] == 1 else f"{name}__{seen[name]}")
    return deduped


def interpolate_processed_csv(input_csv, output_csv, energy_step=0.1):
    df = pd.read_csv(input_csv)
    xy_indices = sorted(
        int(c[1:]) for c in df.columns
        if re.fullmatch(r"x\d+", c) and f"y{c[1:]}" in df.columns
    )
    extra_cols = [c for c in df.columns if c != "file_name" and not re.fullmatch(r"[xy]\d+", c)]

    spectra = []
    global_xmin = -np.inf
    global_xmax = np.inf
    for _, row in df.iterrows():
        xs, ys = [], []
        for i in xy_indices:
            x_val, y_val = row.get(f"x{i}"), row.get(f"y{i}")
            if pd.isna(x_val) or pd.isna(y_val):
                break
            xs.append(float(x_val)); ys.append(float(y_val))
        xs = np.asarray(xs); ys = np.asarray(ys)
        if len(xs) < 2:
            raise ValueError(f"Spectrum {row['file_name']} has fewer than two valid points")
        spectra.append((str(row["file_name"]), xs, ys, row))
        global_xmin = max(global_xmin, xs.min())
        global_xmax = min(global_xmax, xs.max())

    x_start = np.ceil(global_xmin / energy_step) * energy_step
    x_end = np.floor(global_xmax / energy_step) * energy_step
    decimals = max(0, int(np.ceil(-np.log10(energy_step))))
    common_x = np.round(np.arange(x_start, x_end + energy_step / 2, energy_step), decimals)

    # file_name is not guaranteed unique (e.g. replicate spectra); dedupe so every
    # spectrum keeps its own column instead of silently overwriting an earlier one.
    column_names = _dedupe_names([name for name, _, _, _ in spectra])

    out = pd.DataFrame(index=[f"{x:.1f}" for x in common_x])
    for col_name, (_, xs, ys, _) in zip(column_names, spectra):
        out[col_name] = interp1d(xs, ys, kind="linear", fill_value="extrapolate")(common_x)

    for col in extra_cols:
        out.loc[col] = [row[col] for _, _, _, row in spectra]

    out.index.name = "energy"
    out.to_csv(output_csv)
    return out

if DATA_CSV.exists() and not DATA_INTERPOLATED_CSV.exists():
    interpolated = interpolate_processed_csv(DATA_CSV, DATA_INTERPOLATED_CSV, energy_step=0.1)
    print(f"Generated {DATA_INTERPOLATED_CSV.name}: {interpolated.shape}")
elif DATA_INTERPOLATED_CSV.exists():
    interpolated = pd.read_csv(DATA_INTERPOLATED_CSV, index_col=0)
    print(f"Found existing {DATA_INTERPOLATED_CSV.name}: {interpolated.shape}")
else:
    print("data.csv was not found. Provide data.csv or run Sections 1-4 from raw spectra.")

Found existing data_interpolated.csv: (808, 415)


## 6. Data dictionary

This table defines the columns in `data.csv` and the metadata rows in `data_interpolated.csv`.


In [6]:
data_dictionary = pd.DataFrame([
    {"field": "file_name", "location": "data.csv column / data_interpolated column name", "definition": "Spectrum file name or sample identifier."},
    {"field": "x1, x2, ...", "location": "data.csv columns", "definition": "Photon-energy grid values in eV for each spectrum before common-grid interpolation."},
    {"field": "y1, y2, ...", "location": "data.csv columns", "definition": "Gaussian-smoothed and area-normalized XAS intensity values paired with x1, x2, ..."},
    {"field": "numeric energy rows", "location": "data_interpolated.csv row index", "definition": "Common photon-energy grid; each column is one spectrum intensity at that energy."},
    {"field": "mbl", "location": "metadata", "definition": "Mean bond length target used for regression."},
    {"field": "e_density", "location": "metadata", "definition": "Electron-density descriptor when available; missing values are left as NaN."},
    {"field": "B_rate", "location": "metadata", "definition": "B dopant concentration in percent. Values are 0, 1.39, 2.78, 4.17, 5.56, or 6.95."},
    {"field": "N_rate", "location": "metadata", "definition": "N dopant concentration in percent. Values are 0, 1.39, 2.78, 4.17, 5.56, or 6.95."},
    {"field": "type", "location": "metadata", "definition": "Dopant-family code: 0 = pristine, 1 = BG, 2 = NG."},
    {"field": "ndn", "location": "metadata", "definition": "Integer site/category descriptor retained from the original dataset."},
    {"field": "bader", "location": "metadata", "definition": "Mean Bader charge target used for regression."},
    {"field": "PI", "location": "metadata", "definition": "Precomputed pi* peak-related scalar retained from the original dataset."},
    {"field": "SIGMA", "location": "metadata", "definition": "Precomputed sigma* peak-related scalar retained from the original dataset."},
    {"field": "file_name.1", "location": "metadata", "definition": "Original file-name copy retained during table construction."},
    {"field": "rate", "location": "metadata", "definition": "Total dopant concentration in percent; equals the non-zero dopant rate for BG/NG and 0 for pristine."},
])
data_dictionary


,field,location,definition
0,file_name,data.csv column / data_interpolated column name,Spectrum file name or sample identifier.
1,"x1, x2, ...",data.csv columns,Photon-energy grid values in eV for each spect...
2,"y1, y2, ...",data.csv columns,Gaussian-smoothed and area-normalized XAS inte...
3,numeric energy rows,data_interpolated.csv row index,Common photon-energy grid; each column is one ...
4,mbl,metadata,Mean bond length target used for regression.
5,e_density,metadata,Electron-density descriptor when available; mi...
6,B_rate,metadata,B dopant concentration in percent. Values are ...
7,N_rate,metadata,N dopant concentration in percent. Values are ...
8,type,metadata,"Dopant-family code: 0 = pristine, 1 = BG, 2 = NG."
9,ndn,metadata,Integer site/category descriptor retained from...


## 7. Load `data_interpolated.csv` and construct named features

The following loader separates spectral intensity rows from metadata rows. Feature construction uses named energy windows rather than positional column indices.


In [7]:
import numpy as np
import pandas as pd

# Bootstrap globals so Section 7+ can be run without executing cell 2 first.
if "ENERGY_WINDOWS" not in dir():
    RANDOM_STATE = 42
    ENERGY_WINDOWS = {
        "PI": (265.0, 273.5),
        "SIGMA": (273.5, 283.5),
        "POSTEDGE": (283.5, 290.0),
        "FULL": (265.0, 290.0),
    }
    CONCENTRATION_BINS = [0, 1.39, 2.78, 4.17, 5.56, 6.95, np.inf]
    CONCENTRATION_LABELS = ["0%", "1.39%", "2.78%", "4.17%", "5.56%", "6.95%"]
    TYPE_LABELS = {0: "pristine", 1: "BG", 2: "NG"}
    OUT_DIR = ROOT / "dimensionality_reduction_265_290"


def load_interpolated_dataset(path=DATA_INTERPOLATED_CSV):
    raw = pd.read_csv(path, index_col=0)
    energy_index = pd.to_numeric(raw.index, errors="coerce")
    energy_rows = energy_index.notna()

    spectra_t = raw.loc[energy_rows].apply(pd.to_numeric, errors="raise")
    energies = energy_index[energy_rows].astype(float).to_numpy()
    spectra_names = spectra_t.columns.to_list()
    spectra_matrix = spectra_t.T.to_numpy(dtype=float)

    meta = raw.loc[~energy_rows].T.copy()
    for col in ["mbl", "e_density", "B_rate", "N_rate", "bader", "PI", "SIGMA", "rate"]:
        if col in meta.columns:
            meta[col] = pd.to_numeric(meta[col], errors="coerce")
    for col in ["type", "ndn"]:
        if col in meta.columns:
            meta[col] = pd.to_numeric(meta[col], errors="coerce").astype("Int64")
    meta["spectrum_name"] = spectra_names
    return energies, spectra_names, spectra_matrix, meta


def energy_mask(energies, window_name):
    e_min, e_max = ENERGY_WINDOWS[window_name]
    return (energies >= e_min) & (energies <= e_max)


def intensity_feature_frame(energies, spectra_matrix, spectra_names, window_name):
    mask = energy_mask(energies, window_name)
    cols = [f"I_{e:.1f}eV" for e in energies[mask]]
    return pd.DataFrame(spectra_matrix[:, mask], index=spectra_names, columns=cols)


energies, spectra_names, spectra_matrix, meta = load_interpolated_dataset()
feature_summary = []
for name, (lo, hi) in ENERGY_WINDOWS.items():
    mask = energy_mask(energies, name)
    feature_summary.append({"window": name, "energy_min_eV": lo, "energy_max_eV": hi, "n_intensity_features": int(mask.sum())})

print(f"Loaded {len(spectra_names)} spectra and {len(energies)} common-grid energy points.")
pd.DataFrame(feature_summary)

Loaded 415 spectra and 797 common-grid energy points.


,window,energy_min_eV,energy_max_eV,n_intensity_features
0,PI,265.0,273.5,86
1,SIGMA,273.5,283.5,101
2,POSTEDGE,283.5,290.0,66
3,FULL,265.0,290.0,251


## 8. Standard XANES descriptors

These descriptor definitions make the feature construction auditable. They are extracted independently for each spectrum from the common-grid intensity values.


In [8]:
from scipy.stats import linregress

def _window_values(energies, spectrum, e_min, e_max):
    mask = (energies >= e_min) & (energies <= e_max)
    return energies[mask], spectrum[mask]


def extract_xanes_descriptors(energies, spectrum):
    regions = {
        "pi": ENERGY_WINDOWS["PI"],
        "sigma": ENERGY_WINDOWS["SIGMA"],
        "postedge": ENERGY_WINDOWS["POSTEDGE"],
        "full": ENERGY_WINDOWS["FULL"],
    }
    d = {}
    areas = {}
    for reg, (lo, hi) in regions.items():
        e_w, s_w = _window_values(energies, spectrum, lo, hi)
        if len(e_w) < 2:
            peak_e = peak_h = area = centroid = np.nan
        else:
            peak_idx = int(np.argmax(s_w))
            peak_e = float(e_w[peak_idx])
            peak_h = float(s_w[peak_idx])
            area = float(np.trapz(s_w, e_w))
            centroid = float(np.trapz(s_w * e_w, e_w) / area) if area != 0 else np.nan
        d[f"{reg}_peak_energy"] = peak_e
        d[f"{reg}_peak_height"] = peak_h
        d[f"{reg}_area"] = area
        d[f"{reg}_centroid"] = centroid
        areas[reg] = area

    full_area = areas.get("full", np.nan)
    full_area_valid = np.isfinite(full_area) and full_area != 0
    for reg in ["pi", "sigma", "postedge"]:
        area = areas.get(reg, np.nan)
        d[f"area_ratio_{reg}_full"] = area / full_area if full_area_valid else np.nan

    pi_h = d.get("pi_peak_height", np.nan)
    sigma_h = d.get("sigma_peak_height", np.nan)
    d["height_ratio_pi_sigma"] = pi_h / sigma_h if sigma_h != 0 and np.isfinite(sigma_h) else np.nan

    e_full, s_full = _window_values(energies, spectrum, *ENERGY_WINDOWS["FULL"])
    if len(e_full) >= 2:
        full_max = float(np.max(s_full))
        d["full_mean_intensity"] = float(np.mean(s_full))
        d["full_std_intensity"] = float(np.std(s_full))
        d["height_ratio_pi_full"] = pi_h / full_max if full_max != 0 and np.isfinite(pi_h) else np.nan
        half = full_max / 2.0
        idx = np.where(s_full >= half)[0]
        d["half_max_energy"] = float(e_full[idx[0]]) if len(idx) else np.nan
    else:
        d["full_mean_intensity"] = d["full_std_intensity"] = d["height_ratio_pi_full"] = d["half_max_energy"] = np.nan

    e_pi, s_pi = _window_values(energies, spectrum, *ENERGY_WINDOWS["PI"])
    if len(e_pi) >= 3:
        grad = np.gradient(s_pi, e_pi)
        max_grad_idx = int(np.argmax(grad))
        d["edge_derivative_energy"] = float(e_pi[max_grad_idx])
        d["edge_derivative_max"] = float(grad[max_grad_idx])
    else:
        d["edge_derivative_energy"] = d["edge_derivative_max"] = np.nan

    e_post, s_post = _window_values(energies, spectrum, *ENERGY_WINDOWS["POSTEDGE"])
    if len(e_post) >= 2:
        d["postedge_mean_intensity"] = float(np.mean(s_post))
        d["postedge_slope"] = float(linregress(e_post, s_post).slope)
    else:
        d["postedge_mean_intensity"] = d["postedge_slope"] = np.nan
    return d


DESCRIPTOR_GROUPS = {
    "A_peak": ["pi_peak_energy", "pi_peak_height", "sigma_peak_energy", "sigma_peak_height", "postedge_peak_energy", "postedge_peak_height"],
    "B_area": ["pi_area", "sigma_area", "postedge_area", "full_area", "area_ratio_pi_full", "area_ratio_sigma_full", "area_ratio_postedge_full", "height_ratio_pi_sigma", "height_ratio_pi_full"],
    "C_edge": ["pi_centroid", "sigma_centroid", "postedge_centroid", "full_centroid", "edge_derivative_energy", "edge_derivative_max", "half_max_energy", "postedge_mean_intensity", "postedge_slope"],
}
DESCRIPTOR_GROUPS["D_all"] = sorted({c for cols in DESCRIPTOR_GROUPS.values() for c in cols})

descriptor_df = pd.DataFrame([
    {"spectrum_name": name, **extract_xanes_descriptors(energies, spectra_matrix[i])}
    for i, name in enumerate(spectra_names)
])
descriptor_df.head()


,spectrum_name,pi_peak_energy,pi_peak_height,pi_area,pi_centroid,sigma_peak_energy,sigma_peak_height,sigma_area,sigma_centroid,postedge_peak_energy,...,area_ratio_postedge_full,height_ratio_pi_sigma,full_mean_intensity,full_std_intensity,height_ratio_pi_full,half_max_energy,edge_derivative_energy,edge_derivative_max,postedge_mean_intensity,postedge_slope
0,AA center.dat,271.3,0.074188,0.088274,271.705363,277.8,0.231865,0.491534,279.144614,287.0,...,0.310088,0.319962,0.033548,0.035209,0.319962,277.5,271.0,0.155551,0.040063,-0.000961
1,AA edge.dat,271.3,0.074218,0.088287,271.705364,277.8,0.232068,0.491584,279.144574,287.0,...,0.310083,0.319811,0.033552,0.035216,0.319811,277.5,271.0,0.155621,0.040066,-0.000961
2,AAA_A1 center.dat,271.8,0.082252,0.088595,272.072298,278.3,0.254918,0.524915,279.353174,287.5,...,0.324431,0.322663,0.036244,0.039280,0.322663,278.0,271.5,0.177810,0.045300,-0.000952
3,AAA_A1 edge.dat,271.8,0.082342,0.088567,272.072311,278.3,0.254956,0.524741,279.353136,287.5,...,0.324433,0.322966,0.036232,0.039269,0.322966,278.0,271.5,0.177792,0.045285,-0.000952
4,AAA_A2 center.dat,271.8,0.078403,0.087564,272.077610,278.3,0.252524,0.515499,279.342290,287.9,...,0.331282,0.310476,0.035983,0.038782,0.310476,278.0,271.5,0.166254,0.045861,-0.000129


## 9. Dimensionality reduction on intensity features

This reproduces the PCA/UMAP/t-SNE input matrix used for the manuscript response: only intensity values in the 265-290 eV full window are used. Labels and metadata are not included in the fitted matrix.


In [9]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

try:
    from umap.umap_ import UMAP
except ImportError:
    UMAP = None

full_mask = energy_mask(energies, "FULL")
X_full = spectra_matrix[:, full_mask]
energies_full = energies[full_mask]

pca = PCA(n_components=3, random_state=RANDOM_STATE)
pca_xy = pca.fit_transform(X_full)
print("PCA explained variance ratio:", [round(v * 100, 4) for v in pca.explained_variance_ratio_])

coords = pd.DataFrame({
    "spectrum_name": spectra_names,
    "type": meta["type"].to_numpy() if "type" in meta else np.nan,
    "B_rate": meta["B_rate"].to_numpy() if "B_rate" in meta else np.nan,
    "N_rate": meta["N_rate"].to_numpy() if "N_rate" in meta else np.nan,
    "PCA1": pca_xy[:, 0],
    "PCA2": pca_xy[:, 1],
    "PCA3": pca_xy[:, 2],
})

if UMAP is not None:
    umap_xy = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric="euclidean", random_state=RANDOM_STATE).fit_transform(X_full)
    coords["UMAP1"] = umap_xy[:, 0]
    coords["UMAP2"] = umap_xy[:, 1]

tsne_xy = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=RANDOM_STATE).fit_transform(X_full)
coords["tSNE1"] = tsne_xy[:, 0]
coords["tSNE2"] = tsne_xy[:, 1]

OUT_DIR.mkdir(parents=True, exist_ok=True)
coords.to_csv(OUT_DIR / "dimensionality_reduction_coordinates_265_290.csv", index=False)
coords.head()

PCA explained variance ratio: [40.4238, 16.1715, 10.8661]


c:\Users\yinan\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] 指定されたファイルが見つかりません。
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\yinan\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\yinan\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\yinan\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\yinan\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,


,spectrum_name,type,B_rate,N_rate,PCA1,PCA2,PCA3,tSNE1,tSNE2
0,AA center.dat,0,0.0,0.0,-0.201723,-0.036432,-0.025717,-5.894978,-13.739884
1,AA edge.dat,0,0.0,0.0,-0.201733,-0.036437,-0.025742,-6.055918,-13.390556
2,AAA_A1 center.dat,0,0.0,0.0,-0.343829,-0.029680,0.221957,-28.081261,-23.082342
3,AAA_A1 edge.dat,0,0.0,0.0,-0.343717,-0.029656,0.221869,-28.146393,-22.606884
4,AAA_A2 center.dat,0,0.0,0.0,-0.339490,-0.028198,0.216291,-27.823648,-21.982674


## 9b. Dimensionality reduction plots

Reproduces the exact plotting logic from `run_dimensionality_reduction_265_290.py`: PCA/UMAP/t-SNE scatter plots colored by dopant family and concentration, plus PCA PC1 loading vs. mean-spectrum overlays for each dopant group.

In [11]:
import matplotlib.pyplot as plt

REFERENCE_SPECTRA = {
    "Pristine": ("mono center.dat", "#eb4d00", "mono pristine"),
    "BG": ("mono_1B.dat", "green", "1.39% BG"),
    "NG": ("mono_1N.dat", "blue", "1.39% NG"),
}


def get_spectrum_by_name(x_matrix, meta, spectrum_name):
    names = meta["spectrum_name"].astype(str).to_numpy()
    matches = np.flatnonzero(names == spectrum_name)
    if len(matches) == 0:
        lower_matches = np.flatnonzero(np.char.lower(names) == spectrum_name.lower())
        matches = lower_matches
    if len(matches) == 0:
        available = ", ".join(names[:20])
        raise KeyError(
            f"Could not find reference spectrum '{spectrum_name}' in data_interpolated.csv. "
            f"First available spectra: {available}"
        )
    idx = int(matches[0])
    return x_matrix[idx], names[idx]


def make_plot(df, x_col, y_col, x_label, y_label, out_path):
    fig = plt.figure(figsize=(10.6, 7.6))
    ax = fig.add_axes([0.10, 0.13, 0.58, 0.80])
    ax.set_box_aspect(1)

    type_values = pd.to_numeric(df["type"], errors="coerce").fillna(0).astype(int)
    b_rate = pd.to_numeric(df["B_rate"], errors="coerce").fillna(0.0)
    n_rate = pd.to_numeric(df["N_rate"], errors="coerce").fillna(0.0)
    color_min = float(min(b_rate.min(), n_rate.min()))
    color_max = float(max(b_rate.max(), n_rate.max()))

    pristine = (type_values != 1) & (type_values != 2)
    boron = type_values == 1
    nitrogen = type_values == 2

    if pristine.any():
        ax.scatter(
            df.loc[pristine, x_col],
            df.loc[pristine, y_col],
            c="lightgray",
            marker="s",
            edgecolor="#eb4d00",
            linewidth=0.8,
            alpha=0.68,
            label="Pristine",
            zorder=2,
        )

    b_scatter = None
    if boron.any():
        b_scatter = ax.scatter(
            df.loc[boron, x_col],
            df.loc[boron, y_col],
            c=b_rate[boron],
            cmap=plt.cm.summer,
            marker="o",
            edgecolor="#009900",
            linewidth=0.8,
            alpha=0.84,
            label="Boron-doped",
            vmin=color_min,
            vmax=color_max,
            zorder=3,
        )

    n_scatter = None
    if nitrogen.any():
        n_scatter = ax.scatter(
            df.loc[nitrogen, x_col],
            df.loc[nitrogen, y_col],
            c=n_rate[nitrogen],
            cmap=plt.cm.winter,
            marker="^",
            edgecolor="#008cff",
            linewidth=0.8,
            alpha=0.84,
            label="Nitrogen-doped",
            vmin=color_min,
            vmax=color_max,
            zorder=4,
        )

    if b_scatter is not None:
        cax_b = fig.add_axes([0.76, 0.19, 0.025, 0.68])
        cbar_b = fig.colorbar(b_scatter, cax=cax_b)
        cbar_b.set_label("B concentration (%)", fontsize=18, labelpad=17)
        cbar_b.ax.tick_params(labelsize=14)
        cbar_b.ax.yaxis.set_label_position("right")
    if n_scatter is not None:
        cax_n = fig.add_axes([0.89, 0.19, 0.025, 0.68])
        cbar_n = fig.colorbar(n_scatter, cax=cax_n)
        cbar_n.set_label("N concentration (%)", fontsize=18, labelpad=17)
        cbar_n.ax.tick_params(labelsize=14)
        cbar_n.ax.yaxis.set_label_position("right")

    ax.set_xlabel(x_label, fontsize=22)
    ax.set_ylabel(y_label, fontsize=22)
    ax.tick_params(axis="both", labelsize=17)
    ax.legend(
        loc="best",
        fontsize=17,
        frameon=True,
        edgecolor="#B0B0B0",
    )
    ax.grid(True, color="#DDDDDD", linewidth=0.7, alpha=0.7)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.4)
    fig.savefig(out_path, dpi=600, bbox_inches="tight", pad_inches=0.08)
    plt.close(fig)


def make_pca_loading_plot(
    energies_window,
    spectra_subset,
    group_label,
    reference_spectrum,
    reference_label,
    reference_color,
    out_path,
):
    pca_group = PCA(n_components=1, random_state=RANDOM_STATE)
    pca_group.fit(spectra_subset)
    loading = pca_group.components_[0]

    fig, ax1 = plt.subplots(figsize=(7.2, 6.0))
    ax2 = ax1.twinx()

    loading_zero = float(np.mean(loading))
    spectrum_zero = float(np.mean(reference_spectrum))
    spectrum_shifted = reference_spectrum - spectrum_zero + loading_zero

    ax2.plot(
        energies_window,
        spectrum_shifted,
        color=reference_color,
        alpha=1.0,
        linewidth=3.0,
        linestyle="-",
        label=reference_label,
        zorder=1,
    )
    ax2.get_yaxis().set_visible(False)

    ax1.axhline(0.0, color="#999999", linewidth=1.0, linestyle=":", zorder=0)
    ax1.plot(
        energies_window,
        loading,
        marker="o",
        markersize=4.2,
        linestyle="--",
        linewidth=2.1,
        color="#595959",
        label=f"{group_label} PC1",
        alpha=0.88,
        zorder=3,
    )

    ax1.set_xlabel("Photon Energy (eV)", fontsize=18)
    ax1.set_ylabel("PC1 Loading", fontsize=18)
    ax1.set_xlim(ENERGY_WINDOWS["FULL"][0], ENERGY_WINDOWS["FULL"][1])
    ax1.tick_params(axis="both", which="major", labelsize=13)

    for ax in (ax1, ax2):
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color("black")
            spine.set_linewidth(1.4)

    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    fig.legend(
        handles1 + handles2,
        labels1 + labels2,
        loc="upper right",
        bbox_to_anchor=(0.96, 0.94),
        ncol=1,
        fontsize=12,
        frameon=False,
    )
    fig.tight_layout()
    fig.savefig(out_path, dpi=600)
    plt.close(fig)


pca_x_label = f"PCA1 ({pca.explained_variance_ratio_[0]*100:.1f}%)"
pca_y_label = f"PCA2 ({pca.explained_variance_ratio_[1]*100:.1f}%)"
make_plot(coords, "PCA1", "PCA2", pca_x_label, pca_y_label, OUT_DIR / "pca_265_290.png")
if "UMAP1" in coords.columns:
    make_plot(coords, "UMAP1", "UMAP2", "UMAP1", "UMAP2", OUT_DIR / "umap_265_290.png")
else:
    print("Skipping UMAP plot: umap-learn is not installed.")
make_plot(coords, "tSNE1", "tSNE2", "t-SNE1", "t-SNE2", OUT_DIR / "tsne_265_290.png")

type_values_arr = pd.to_numeric(meta["type"], errors="coerce").fillna(0).astype(int).to_numpy()
b_rate_arr = pd.to_numeric(meta["B_rate"], errors="coerce").fillna(0.0).to_numpy()
n_rate_arr = pd.to_numeric(meta["N_rate"], errors="coerce").fillna(0.0).to_numpy()
group_masks = [
    ("Pristine", (type_values_arr != 1) & (type_values_arr != 2), "pca_pristine_pc1_loading_vs_mean_spectrum_265_290.png"),
    ("BG", b_rate_arr > 0, "pca_BG_pc1_loading_vs_mean_spectrum_265_290.png"),
    ("NG", n_rate_arr > 0, "pca_NG_pc1_loading_vs_mean_spectrum_265_290.png"),
]
for group_label, mask, filename in group_masks:
    if int(mask.sum()) < 2:
        print(f"Skipping {group_label}: fewer than 2 spectra.")
        continue
    reference_name, reference_color, reference_legend_label = REFERENCE_SPECTRA[group_label]
    reference_spectrum, matched_name = get_spectrum_by_name(X_full, meta, reference_name)
    make_pca_loading_plot(
        energies_full,
        X_full[mask],
        group_label,
        reference_spectrum,
        reference_legend_label,
        reference_color,
        OUT_DIR / filename,
    )
    print(f"{group_label}: PCA loading fitted with {int(mask.sum())} spectra; reference spectrum: {matched_name}.")

print(f"Saved plots in: {OUT_DIR}")

Pristine: PCA loading fitted with 36 spectra; reference spectrum: mono center.dat.
BG: PCA loading fitted with 191 spectra; reference spectrum: mono_1B.dat.
NG: PCA loading fitted with 188 spectra; reference spectrum: mono_1N.dat.
Saved plots in: c:\UTokyo\Mizoguchi_lab\Kotsugi_paper\public_workflow_validation\dimensionality_reduction_265_290


## 10. Split definitions used in ML experiments

Public scripts should state the split protocol explicitly. The analysis used two protocols:

- Spectrum-level 80/20 split stratified by dopant family and concentration.
- 5-fold `StratifiedGroupKFold` by atomic configuration ID, so spectra from the same configuration are kept in the same fold.


In [25]:
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

def parse_configuration_id(name):
    # Strip .dat or .dat.N (deduplicated names like mono_1B.dat.1) before matching
    base = re.sub(r"\.dat(\.\d+)?$", "", str(name).strip())
    if base.startswith("mono_"):
        match = re.match(r"^(mono_\d+[A-Za-z]+\d*)(_\(\d+\))?$", base)
        if match:
            return match.group(1)
    if base in ["mono center", "mono edge"]:
        return "mono"
    match = re.match(r"^(\S+)\s+(center|edge)(_\(\d+\))?$", base)
    if match:
        return match.group(1).split("_")[0]
    return base


def add_split_columns(meta):
    out = meta.copy()

    # Use file_name.1 (true filename) when available, fall back to spectrum_name.
    # Deduplicated spectra like mono_1B.dat.1 must map to the same configuration_id
    # as mono_1B.dat — done via file_name.1 column.
    name_source = (
        out["file_name.1"].astype(str)
        if "file_name.1" in out.columns
        else out["spectrum_name"].astype(str)
    )
    out["configuration_id"] = [parse_configuration_id(n) for n in name_source]
    out["dopant_family"] = out["type"].astype(int).map(TYPE_LABELS)

    # Parse concentration from filename for the spectrum-level stratified split.
    # (Pristine spectra have rate=6.95 in the CSV but their true concentration is 0.0.)
    conc_map = {1: 1.39, 2: 2.78, 3: 4.17, 4: 5.56, 5: 6.95}
    parsed_groups = []
    parsed_concs = []
    for name in out["spectrum_name"]:
        m = re.match(r"mono_(\d+)([BN])", str(name).strip())
        if m:
            g = "B" if m.group(2) == "B" else "N"
            c = conc_map.get(int(m.group(1)), 0.0)
        else:
            g, c = "pristine", 0.0
        parsed_groups.append(g)
        parsed_concs.append(c)
    out["parsed_group"] = parsed_groups
    out["concentration"] = parsed_concs
    out["stratify_key"] = out["parsed_group"] + "_" + pd.Series(parsed_concs, index=out.index).astype(str)

    # stratify_key_rate uses the 'rate' column (as in rf_results_v2/classification_grouped_config),
    # which is used for the grouped CV fold assignment.
    rate_col = out["rate"].astype(float) if "rate" in out.columns else pd.Series(parsed_concs, index=out.index)
    out["stratify_key_rate"] = out["dopant_family"].astype(str) + "_" + rate_col.astype(str)

    return out


def spectrum_level_split(meta_with_splits, target_col="mbl"):
    # rf_results_v2 splits on ALL 415 spectra (no valid_idx filtering by target NaN)
    stratify = meta_with_splits["stratify_key"].to_numpy()
    train_local, test_local = train_test_split(
        np.arange(len(meta_with_splits)), test_size=0.2, random_state=RANDOM_STATE, stratify=stratify
    )
    return train_local, test_local


def grouped_cv_folds(meta_with_splits, target_col="mbl", n_splits=5):
    # rf_results_v2/classification_grouped_config uses rate-based stratify_key and all 415 spectra
    groups = meta_with_splits["configuration_id"].to_numpy()
    stratify = meta_with_splits["stratify_key_rate"].to_numpy()
    fold_assignment = np.full(len(meta_with_splits), -1, dtype=int)
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    for fold_id, (_, val_local) in enumerate(sgkf.split(np.zeros(len(meta_with_splits)), stratify, groups)):
        fold_assignment[val_local] = fold_id
    return fold_assignment

meta_split = add_split_columns(meta)
train_idx, test_idx = spectrum_level_split(meta_split)
fold_assignment = grouped_cv_folds(meta_split)
print(f"Spectrum-level split: {len(train_idx)} train / {len(test_idx)} test")
print(pd.Series(fold_assignment[fold_assignment >= 0]).value_counts().sort_index())

# Sanity checks
pristine_concs = meta_split.loc[meta_split["parsed_group"] == "pristine", "concentration"].unique()
assert list(pristine_concs) == [0.0], f"Bug: pristine concentration should be [0.0], got {pristine_concs}"
print(f"\nPristine concentration check passed: {pristine_concs}")

for dedup_name, expected_cid in [("mono_1B.dat.1", "mono_1B"), ("mono_1B.dat.2", "mono_1B"),
                                  ("mono_1N.dat.1", "mono_1N"), ("mono_1N.dat.2", "mono_1N")]:
    rows = meta_split[meta_split["spectrum_name"] == dedup_name]
    if not rows.empty:
        cid = rows["configuration_id"].iloc[0]
        assert cid == expected_cid, f"Bug: {dedup_name} got configuration_id={cid}, expected {expected_cid}"
        print(f"Config-id check passed: {dedup_name} -> {cid}")


Spectrum-level split: 332 train / 83 test
0    104
1     89
2     73
3     75
4     74
Name: count, dtype: int64

Pristine concentration check passed: [0.]


c:\Users\yinan\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:994: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


## 11. Random Forest manuscript experiments

This section condenses the Random Forest scripts in `rf_results_v2/` and `regression_results_grouped_config_v2/` into auditable notebook functions.

It covers:

- RF regression for `mbl` and `bader`
- RF classification for `B_rate` and `N_rate` concentration classes
- spectrum-level stratified 80/20 evaluation
- configuration-level grouped 5-fold cross-validation
- named energy-window feature selection


In [26]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, StratifiedKFold

RF_REG_PARAMS = {"n_estimators": 100, "random_state": RANDOM_STATE}
# rf_results_v2/classification/classification.py used random_state=0 for the classifier
RF_CLS_PARAMS = {"n_estimators": 100, "random_state": 0}

TARGET_INFO = {
    "mbl": {"label": "Mean Bond Length", "unit": "Ang."},
    "bader": {"label": "Mean Bader Charge", "unit": "e"},
}
DOPANT_TARGET = {"B": "B_rate", "N": "N_rate"}


def regression_metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": mean_absolute_error(y_true, y_pred),
    }


def matrix_for_window(window_name):
    mask = energy_mask(energies, window_name)
    return spectra_matrix[:, mask], energies[mask]


def rf_regression_stratified(target, window_name):
    X_all, feature_energies = matrix_for_window(window_name)
    y_all = meta_split[target].to_numpy(dtype=float)
    train_idx, test_idx = spectrum_level_split(meta_split, target_col=target)

    X_train, X_test = X_all[train_idx], X_all[test_idx]
    y_train, y_test = y_all[train_idx], y_all[test_idx]

    cv_r2, cv_mae = [], []
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    for tr, va in kf.split(X_train):
        model = RandomForestRegressor(**RF_REG_PARAMS)
        model.fit(X_train[tr], y_train[tr])
        pred = model.predict(X_train[va])
        cv_r2.append(r2_score(y_train[va], pred))
        cv_mae.append(mean_absolute_error(y_train[va], pred))

    model = RandomForestRegressor(**RF_REG_PARAMS)
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    test_metrics = regression_metrics(y_test, test_pred)
    summary = {
        "Model": "RF", "Split": "stratified", "Task": "regression", "Target": target,
        "Range": window_name, "N_features": X_train.shape[1],
        "CV_R2_mean": float(np.mean(cv_r2)), "CV_R2_std": float(np.std(cv_r2)),
        "CV_MAE_mean": float(np.mean(cv_mae)), "CV_MAE_std": float(np.std(cv_mae)),
        "Test_R2": test_metrics["R2"], "Test_RMSE": test_metrics["RMSE"], "Test_MAE": test_metrics["MAE"],
    }
    test_predictions = pd.DataFrame({
        "spectrum_name": meta_split.iloc[test_idx]["spectrum_name"].to_numpy(),
        "type": meta_split.iloc[test_idx]["type"].to_numpy(),
        "y_true": y_test,
        "y_pred": test_pred,
    })
    train_predictions = pd.DataFrame({
        "spectrum_name": meta_split.iloc[train_idx]["spectrum_name"].to_numpy(),
        "type": meta_split.iloc[train_idx]["type"].to_numpy(),
        "y_true": y_train,
        "y_pred": train_pred,
    })
    feature_importance = pd.DataFrame({"energy_eV": feature_energies, "importance": model.feature_importances_})
    return summary, test_predictions, train_predictions, feature_importance


def rf_regression_grouped(target, window_name):
    X_all, feature_energies = matrix_for_window(window_name)
    y_all = meta_split[target].to_numpy(dtype=float)
    fold_assignment = grouped_cv_folds(meta_split, target_col=target, n_splits=5)
    valid_idx = np.where(fold_assignment >= 0)[0]
    X_valid = X_all[valid_idx]
    y_valid = y_all[valid_idx]
    folds = fold_assignment[valid_idx]

    oof_pred = np.full(len(valid_idx), np.nan)
    fold_rows = []
    importances = []
    for fold_id in sorted(np.unique(folds)):
        test_mask = folds == fold_id
        train_mask = ~test_mask
        model = RandomForestRegressor(**RF_REG_PARAMS)
        model.fit(X_valid[train_mask], y_valid[train_mask])
        pred = model.predict(X_valid[test_mask])
        oof_pred[test_mask] = pred
        m = regression_metrics(y_valid[test_mask], pred)
        fold_rows.append({"fold": int(fold_id), **m})
        importances.append(model.feature_importances_)

    oof_metrics = regression_metrics(y_valid, oof_pred)
    fold_df = pd.DataFrame(fold_rows)
    summary = {
        "Model": "RF", "Split": "grouped", "Task": "regression", "Target": target,
        "Range": window_name, "N_features": X_valid.shape[1],
        "CV_R2_mean": fold_df["R2"].mean(), "CV_R2_std": fold_df["R2"].std(ddof=0),
        "CV_MAE_mean": fold_df["MAE"].mean(), "CV_MAE_std": fold_df["MAE"].std(ddof=0),
        "OOF_R2": oof_metrics["R2"], "OOF_RMSE": oof_metrics["RMSE"], "OOF_MAE": oof_metrics["MAE"],
    }
    oof_predictions = pd.DataFrame({
        "spectrum_name": meta_split.iloc[valid_idx]["spectrum_name"].to_numpy(),
        "configuration_id": meta_split.iloc[valid_idx]["configuration_id"].to_numpy(),
        "fold_id": folds,
        "type": meta_split.iloc[valid_idx]["type"].to_numpy(),
        "y_true": y_valid,
        "y_pred": oof_pred,
    })
    imp_arr = np.asarray(importances)
    feature_importance = pd.DataFrame({
        "energy_eV": feature_energies,
        "importance_mean": imp_arr.mean(axis=0),
        "importance_std": imp_arr.std(axis=0),
    })
    return summary, fold_df, oof_predictions, feature_importance


def concentration_classes(rate_values):
    return np.digitize(np.asarray(rate_values, dtype=float), CONCENTRATION_BINS) - 1


def rf_classification_stratified(dopant, window_name):
    X_all, feature_energies = matrix_for_window(window_name)
    y_all = concentration_classes(meta_split[DOPANT_TARGET[dopant]].fillna(0.0).to_numpy())
    train_idx, test_idx = spectrum_level_split(meta_split, target_col="mbl")
    X_train, X_test = X_all[train_idx], X_all[test_idx]
    y_train, y_test = y_all[train_idx], y_all[test_idx]

    cv_acc, cv_f1 = [], []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    for tr, va in skf.split(X_train, y_train):
        model = RandomForestClassifier(**RF_CLS_PARAMS)
        model.fit(X_train[tr], y_train[tr])
        pred = model.predict(X_train[va])
        cv_acc.append(accuracy_score(y_train[va], pred))
        cv_f1.append(f1_score(y_train[va], pred, average="weighted"))

    model = RandomForestClassifier(**RF_CLS_PARAMS)
    model.fit(X_train, y_train)
    test_pred = model.predict(X_test)
    summary = {
        "Model": "RF", "Split": "stratified", "Task": "classification", "Target": DOPANT_TARGET[dopant],
        "Range": window_name, "N_features": X_train.shape[1],
        "CV_Accuracy_mean": float(np.mean(cv_acc)), "CV_Accuracy_std": float(np.std(cv_acc)),
        "CV_F1_mean": float(np.mean(cv_f1)), "CV_F1_std": float(np.std(cv_f1)),
        "Test_Accuracy": accuracy_score(y_test, test_pred),
        "Test_F1": f1_score(y_test, test_pred, average="weighted"),
    }
    predictions = pd.DataFrame({
        "spectrum_name": meta_split.iloc[test_idx]["spectrum_name"].to_numpy(),
        "y_true_class": y_test,
        "y_pred_class": test_pred,
    })
    feature_importance = pd.DataFrame({"energy_eV": feature_energies, "importance": model.feature_importances_})
    return summary, predictions, feature_importance


def rf_classification_grouped(dopant, window_name):
    X_all, feature_energies = matrix_for_window(window_name)
    y_all = concentration_classes(meta_split[DOPANT_TARGET[dopant]].fillna(0.0).to_numpy())
    fold_assignment = grouped_cv_folds(meta_split, target_col="mbl", n_splits=5)
    valid_idx = np.where(fold_assignment >= 0)[0]
    X_valid = X_all[valid_idx]
    y_valid = y_all[valid_idx]
    folds = fold_assignment[valid_idx]

    oof_pred = np.full(len(valid_idx), -1, dtype=int)
    fold_rows = []
    importances = []
    for fold_id in sorted(np.unique(folds)):
        test_mask = folds == fold_id
        train_mask = ~test_mask
        model = RandomForestClassifier(**RF_CLS_PARAMS)
        model.fit(X_valid[train_mask], y_valid[train_mask])
        pred = model.predict(X_valid[test_mask])
        oof_pred[test_mask] = pred
        fold_rows.append({
            "fold": int(fold_id),
            "Accuracy": accuracy_score(y_valid[test_mask], pred),
            "F1": f1_score(y_valid[test_mask], pred, average="weighted"),
        })
        importances.append(model.feature_importances_)

    fold_df = pd.DataFrame(fold_rows)
    summary = {
        "Model": "RF", "Split": "grouped", "Task": "classification", "Target": DOPANT_TARGET[dopant],
        "Range": window_name, "N_features": X_valid.shape[1],
        "CV_Accuracy_mean": fold_df["Accuracy"].mean(), "CV_Accuracy_std": fold_df["Accuracy"].std(ddof=0),
        "CV_F1_mean": fold_df["F1"].mean(), "CV_F1_std": fold_df["F1"].std(ddof=0),
        "OOF_Accuracy": accuracy_score(y_valid, oof_pred),
        "OOF_F1": f1_score(y_valid, oof_pred, average="weighted"),
    }
    oof_predictions = pd.DataFrame({
        "spectrum_name": meta_split.iloc[valid_idx]["spectrum_name"].to_numpy(),
        "configuration_id": meta_split.iloc[valid_idx]["configuration_id"].to_numpy(),
        "fold_id": folds,
        "y_true_class": y_valid,
        "y_pred_class": oof_pred,
    })
    imp_arr = np.asarray(importances)
    feature_importance = pd.DataFrame({
        "energy_eV": feature_energies,
        "importance_mean": imp_arr.mean(axis=0),
        "importance_std": imp_arr.std(axis=0),
    })
    return summary, fold_df, oof_predictions, feature_importance


In [27]:
RUN_RF_EXPERIMENTS = True

if RUN_RF_EXPERIMENTS:
    rf_rows = []
    for target in ["mbl", "bader"]:
        for window in ENERGY_WINDOWS:
            row, *_ = rf_regression_stratified(target, window)
            rf_rows.append(row)
            row, *_ = rf_regression_grouped(target, window)
            rf_rows.append(row)
    for dopant in ["B", "N"]:
        for window in ENERGY_WINDOWS:
            row, *_ = rf_classification_stratified(dopant, window)
            rf_rows.append(row)
            row, *_ = rf_classification_grouped(dopant, window)
            rf_rows.append(row)
    rf_summary = pd.DataFrame(rf_rows)
    display(rf_summary)
else:
    print("RF experiment functions are defined. Set RUN_RF_EXPERIMENTS = True to run all RF summaries.")


c:\Users\yinan\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:994: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\yinan\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:994: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\yinan\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:994: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\yinan\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:994: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\yinan\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:994: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\yinan\anaconda3\Lib\s

,Model,Split,Task,Target,Range,N_features,CV_R2_mean,CV_R2_std,CV_MAE_mean,CV_MAE_std,...,OOF_RMSE,OOF_MAE,CV_Accuracy_mean,CV_Accuracy_std,CV_F1_mean,CV_F1_std,Test_Accuracy,Test_F1,OOF_Accuracy,OOF_F1
0,RF,stratified,regression,mbl,PI,86,0.973301,0.012302,0.003113,0.000434,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RF,grouped,regression,mbl,PI,86,0.969452,0.009421,0.003527,0.000726,...,0.006272,0.003558,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,RF,stratified,regression,mbl,SIGMA,101,0.940120,0.036822,0.004470,0.000578,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,RF,grouped,regression,mbl,SIGMA,101,0.877355,0.088982,0.005870,0.001525,...,0.012406,0.005894,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RF,stratified,regression,mbl,POSTEDGE,66,0.941230,0.018388,0.004720,0.000383,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,RF,grouped,regression,mbl,POSTEDGE,66,0.922560,0.031021,0.005113,0.000620,...,0.009857,0.005181,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,RF,stratified,regression,mbl,FULL,251,0.978506,0.006108,0.002986,0.000408,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,RF,grouped,regression,mbl,FULL,251,0.972932,0.008762,0.003477,0.000743,...,0.005928,0.003520,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,RF,stratified,regression,bader,PI,86,0.981762,0.013661,0.080144,0.019822,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,RF,grouped,regression,bader,PI,86,0.980193,0.006805,0.086288,0.019706,...,0.196995,0.087048,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 12. Baseline models: RF, XGBoost, Lasso, and MLP

This section summarizes the baseline-model logic from `ml_experiments/`. The same named windows and split definitions are reused for all models.

The Lasso classification baseline uses fixed-`C` L1 logistic regression (`C = 1.0`) instead of `LogisticRegressionCV`; this avoids unstable multi-class coefficient-shape behavior in small grouped folds while preserving the intended sparse linear baseline.


In [35]:
from sklearn.linear_model import Lasso, LassoCV, LogisticRegression
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_is_fitted

try:
    import xgboost as xgb
except ImportError:
    xgb = None

MODEL_NAMES = ["RF", "XGB", "Lasso", "MLP"]

XGB_REG_PARAMS = dict(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, verbosity=0, n_jobs=1,
)
XGB_CLS_PARAMS = dict(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, verbosity=0, n_jobs=1,
    eval_metric="mlogloss",
)
MLP_REG_PARAMS = dict(
    hidden_layer_sizes=(128, 64), activation="relu", solver="adam",
    max_iter=5000, early_stopping=True, validation_fraction=0.15,
    learning_rate_init=0.001, random_state=RANDOM_STATE,
)
MLP_CLS_PARAMS = dict(
    hidden_layer_sizes=(128, 64), activation="relu", solver="adam",
    max_iter=2000, early_stopping=True, validation_fraction=0.15,
    learning_rate_init=0.001, random_state=RANDOM_STATE,
)


def fit_predict_regression_baseline(model_name, X_train, y_train, X_test):
    if model_name == "RF":
        model = RandomForestRegressor(**RF_REG_PARAMS)
        model.fit(X_train, y_train)
        return model.predict(X_test), {"model": model}

    if model_name == "XGB":
        if xgb is None:
            raise ImportError("xgboost is not installed")
        model = xgb.XGBRegressor(**XGB_REG_PARAMS)
        model.fit(X_train, y_train)
        return model.predict(X_test), {"model": model}

    if model_name == "Lasso":
        x_scaler = StandardScaler()
        y_scaler = StandardScaler()
        X_train_s = x_scaler.fit_transform(X_train)
        X_test_s = x_scaler.transform(X_test)
        y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
        lasso_cv = LassoCV(cv=5, max_iter=50000, random_state=RANDOM_STATE)
        lasso_cv.fit(X_train_s, y_train_s)
        model = Lasso(alpha=lasso_cv.alpha_, max_iter=50000)
        model.fit(X_train_s, y_train_s)
        pred_s = model.predict(X_test_s)
        pred = y_scaler.inverse_transform(pred_s.reshape(-1, 1)).ravel()
        return pred, {"model": model, "x_scaler": x_scaler, "y_scaler": y_scaler, "alpha": lasso_cv.alpha_}

    if model_name == "MLP":
        x_scaler = StandardScaler()
        y_scaler = StandardScaler()
        X_train_s = x_scaler.fit_transform(X_train)
        X_test_s = x_scaler.transform(X_test)
        y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
        model = MLPRegressor(**MLP_REG_PARAMS)
        model.fit(X_train_s, y_train_s)
        pred_s = model.predict(X_test_s)
        pred = y_scaler.inverse_transform(pred_s.reshape(-1, 1)).ravel()
        return pred, {"model": model, "x_scaler": x_scaler, "y_scaler": y_scaler}

    raise ValueError(model_name)


def fit_predict_classification_baseline(model_name, X_train, y_train, X_test):
    if model_name == "RF":
        model = RandomForestClassifier(**RF_CLS_PARAMS)
        model.fit(X_train, y_train)
        return model.predict(X_test), {"model": model}

    if model_name == "XGB":
        if xgb is None:
            raise ImportError("xgboost is not installed")
        classes = np.array(sorted(set(y_train)))
        class_to_int = {c: i for i, c in enumerate(classes)}
        int_to_class = {i: c for c, i in class_to_int.items()}
        y_train_enc = np.array([class_to_int[c] for c in y_train])
        model = xgb.XGBClassifier(**XGB_CLS_PARAMS, num_class=len(classes))
        model.fit(X_train, y_train_enc)
        pred_enc = model.predict(X_test)
        pred = np.array([int_to_class[int(v)] for v in pred_enc])
        return pred, {"model": model, "classes": classes}

    if model_name == "Lasso":
        scaler = StandardScaler()
        model = LogisticRegression(
            penalty="l1", C=1.0, solver="saga", max_iter=5000,
            random_state=RANDOM_STATE, n_jobs=1,
        )
        model.fit(scaler.fit_transform(X_train), y_train)
        return model.predict(scaler.transform(X_test)), {"model": model, "x_scaler": scaler, "C": 1.0}

    if model_name == "MLP":
        scaler = StandardScaler()
        model = MLPClassifier(**MLP_CLS_PARAMS)
        model.fit(scaler.fit_transform(X_train), y_train)
        return model.predict(scaler.transform(X_test)), {"model": model, "x_scaler": scaler}

    raise ValueError(model_name)


def _cv_regression(model_name, X_train, y_train):
    """5-fold KFold CV on training set, returns (cv_r2_mean, cv_r2_std, cv_mae_mean, cv_mae_std)."""
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_r2, cv_mae = [], []
    for tr, va in kf.split(X_train):
        pred, _ = fit_predict_regression_baseline(model_name, X_train[tr], y_train[tr], X_train[va])
        cv_r2.append(r2_score(y_train[va], pred))
        cv_mae.append(mean_absolute_error(y_train[va], pred))
    return float(np.mean(cv_r2)), float(np.std(cv_r2, ddof=0)), float(np.mean(cv_mae)), float(np.std(cv_mae, ddof=0))


def _cv_classification(model_name, X_train, y_train):
    """5-fold StratifiedKFold CV on training set, returns (cv_acc_mean, cv_acc_std, cv_f1_mean, cv_f1_std)."""
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_acc, cv_f1 = [], []
    for tr, va in skf.split(X_train, y_train):
        pred, _ = fit_predict_classification_baseline(model_name, X_train[tr], y_train[tr], X_train[va])
        cv_acc.append(accuracy_score(y_train[va], pred))
        cv_f1.append(f1_score(y_train[va], pred, average="weighted", zero_division=0))
    return float(np.mean(cv_acc)), float(np.std(cv_acc, ddof=0)), float(np.mean(cv_f1)), float(np.std(cv_f1, ddof=0))


def evaluate_regression_baseline(model_name, split, target, window_name):
    X_all, _ = matrix_for_window(window_name)
    y_all = meta_split[target].to_numpy(dtype=float)

    if split == "stratified":
        train_idx, test_idx = spectrum_level_split(meta_split, target_col=target)
        X_tr, X_te = X_all[train_idx], X_all[test_idx]
        y_tr, y_te = y_all[train_idx], y_all[test_idx]
        cv_r2_mean, cv_r2_std, cv_mae_mean, cv_mae_std = _cv_regression(model_name, X_tr, y_tr)
        pred, aux = fit_predict_regression_baseline(model_name, X_tr, y_tr, X_te)
        m = regression_metrics(y_te, pred)
        row = {"Model": model_name, "Split": split, "Task": "regression", "Target": target, "Range": window_name,
               "N_features": X_all.shape[1],
               "CV_R2_mean": cv_r2_mean, "CV_R2_std": cv_r2_std,
               "CV_MAE_mean": cv_mae_mean, "CV_MAE_std": cv_mae_std,
               "Test_R2": m["R2"], "Test_RMSE": m["RMSE"], "Test_MAE": m["MAE"]}
        if "alpha" in aux:
            row["Alpha"] = aux["alpha"]
        return row

    if split == "grouped":
        fold_assignment = grouped_cv_folds(meta_split, target_col=target, n_splits=5)
        valid_idx = np.where(fold_assignment >= 0)[0]
        X_valid = X_all[valid_idx]
        y_valid = y_all[valid_idx]
        folds = fold_assignment[valid_idx]
        oof_pred = np.full(len(valid_idx), np.nan)
        fold_rows = []
        alphas = []
        for fold_id in sorted(np.unique(folds)):
            te = folds == fold_id
            tr = ~te
            pred, aux = fit_predict_regression_baseline(model_name, X_valid[tr], y_valid[tr], X_valid[te])
            oof_pred[te] = pred
            fold_rows.append(regression_metrics(y_valid[te], pred))
            if "alpha" in aux:
                alphas.append(aux["alpha"])
        fold_df = pd.DataFrame(fold_rows)
        oof_m = regression_metrics(y_valid, oof_pred)
        row = {"Model": model_name, "Split": split, "Task": "regression", "Target": target, "Range": window_name,
               "N_features": X_valid.shape[1],
               "CV_R2_mean": fold_df["R2"].mean(), "CV_R2_std": fold_df["R2"].std(ddof=0),
               "CV_MAE_mean": fold_df["MAE"].mean(), "CV_MAE_std": fold_df["MAE"].std(ddof=0),
               "OOF_R2": oof_m["R2"], "OOF_RMSE": oof_m["RMSE"], "OOF_MAE": oof_m["MAE"]}
        if alphas:
            row["Alpha_mean"] = float(np.mean(alphas))
        return row

    raise ValueError(split)


def evaluate_classification_baseline(model_name, split, dopant, window_name):
    X_all, _ = matrix_for_window(window_name)
    y_all = concentration_classes(meta_split[DOPANT_TARGET[dopant]].fillna(0.0).to_numpy())

    if split == "stratified":
        train_idx, test_idx = spectrum_level_split(meta_split, target_col="mbl")
        X_tr, X_te = X_all[train_idx], X_all[test_idx]
        y_tr, y_te = y_all[train_idx], y_all[test_idx]
        cv_acc_mean, cv_acc_std, cv_f1_mean, cv_f1_std = _cv_classification(model_name, X_tr, y_tr)
        pred, _ = fit_predict_classification_baseline(model_name, X_tr, y_tr, X_te)
        return {"Model": model_name, "Split": split, "Task": "classification", "Target": DOPANT_TARGET[dopant],
                "Range": window_name, "N_features": X_all.shape[1],
                "CV_Accuracy_mean": cv_acc_mean, "CV_Accuracy_std": cv_acc_std,
                "CV_F1_mean": cv_f1_mean, "CV_F1_std": cv_f1_std,
                "Test_Accuracy": accuracy_score(y_te, pred),
                "Test_F1": f1_score(y_te, pred, average="weighted", zero_division=0)}

    if split == "grouped":
        fold_assignment = grouped_cv_folds(meta_split, target_col="mbl", n_splits=5)
        valid_idx = np.where(fold_assignment >= 0)[0]
        X_valid = X_all[valid_idx]
        y_valid = y_all[valid_idx]
        folds = fold_assignment[valid_idx]
        oof_pred = np.full(len(valid_idx), -1, dtype=int)
        fold_rows = []
        for fold_id in sorted(np.unique(folds)):
            te = folds == fold_id
            tr = ~te
            pred, _ = fit_predict_classification_baseline(model_name, X_valid[tr], y_valid[tr], X_valid[te])
            oof_pred[te] = pred
            fold_rows.append({"Accuracy": accuracy_score(y_valid[te], pred), "F1": f1_score(y_valid[te], pred, average="weighted")})
        fold_df = pd.DataFrame(fold_rows)
        return {"Model": model_name, "Split": split, "Task": "classification", "Target": DOPANT_TARGET[dopant],
                "Range": window_name, "N_features": X_valid.shape[1],
                "CV_Accuracy_mean": fold_df["Accuracy"].mean(), "CV_F1_mean": fold_df["F1"].mean(),
                "OOF_Accuracy": accuracy_score(y_valid, oof_pred),
                "OOF_F1": f1_score(y_valid, oof_pred, average="weighted")}

    raise ValueError(split)


In [32]:
RUN_BASELINE_EXPERIMENTS = True

if RUN_BASELINE_EXPERIMENTS:
    baseline_rows = []
    for model_name in MODEL_NAMES:
        if model_name == "XGB" and xgb is None:
            print("Skipping XGB because xgboost is not installed")
            continue
        for split in ["stratified", "grouped"]:
            for target in ["mbl", "bader"]:
                for window in ENERGY_WINDOWS:
                    baseline_rows.append(evaluate_regression_baseline(model_name, split, target, window))
            for dopant in ["B", "N"]:
                for window in ENERGY_WINDOWS:
                    baseline_rows.append(evaluate_classification_baseline(model_name, split, dopant, window))
    baseline_summary = pd.DataFrame(baseline_rows)

    # Save to CSV for easy inspection outside the notebook
    _out_csv = ROOT / "baseline_summary.csv"
    baseline_summary.to_csv(_out_csv, index=False)
    print(f"Saved baseline summary to {_out_csv}")

    display(baseline_summary)
else:
    print("Baseline functions are defined. Set RUN_BASELINE_EXPERIMENTS = True to run RF/XGB/Lasso/MLP summaries.")


c:\Users\yinan\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\yinan\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\yinan\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\yinan\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\yinan\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\yinan\anaconda3\Lib\s

Saved baseline summary to c:\UTokyo\Mizoguchi_lab\Kotsugi_paper\public_workflow_validation\baseline_summary.csv


,Model,Split,Task,Target,Range,N_features,CV_R2_mean,CV_R2_std,CV_MAE_mean,CV_MAE_std,...,CV_F1_std,Test_Accuracy,Test_F1,OOF_R2,OOF_RMSE,OOF_MAE,OOF_Accuracy,OOF_F1,Alpha,Alpha_mean
0,RF,stratified,regression,mbl,PI,86,0.973301,0.012302,0.003113,0.000434,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RF,stratified,regression,mbl,SIGMA,101,0.940120,0.036822,0.004470,0.000578,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,RF,stratified,regression,mbl,POSTEDGE,66,0.941230,0.018388,0.004720,0.000383,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,RF,stratified,regression,mbl,FULL,251,0.978506,0.006108,0.002986,0.000408,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RF,stratified,regression,bader,PI,86,0.981762,0.013661,0.080144,0.019822,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,MLP,grouped,classification,B_rate,FULL,251,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.746988,0.740092,NaN,NaN
124,MLP,grouped,classification,N_rate,PI,86,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.749398,0.718332,NaN,NaN
125,MLP,grouped,classification,N_rate,SIGMA,101,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.693976,0.673580,NaN,NaN
126,MLP,grouped,classification,N_rate,POSTEDGE,66,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.761446,0.736677,NaN,NaN


## 13. Expected outputs

Running this notebook can produce or audit the following public outputs:

| Output | Description |
|---|---|
| `processed_xas/smoothed/*.dat` | Gaussian-smoothed spectra, sigma = 1 grid point. |
| `processed_xas/normalized/*.dat` | Area-normalized spectra using Simpson integration. |
| `data.csv` | Per-spectrum wide table with x/y pairs and metadata. |
| `data_interpolated.csv` | Common-grid table used for PCA/UMAP/t-SNE and ML experiments. |
| `dimensionality_reduction_265_290/dimensionality_reduction_coordinates_265_290.csv` | PCA/UMAP/t-SNE coordinates from 265-290 eV intensity features. |
| `xanes_descriptors.csv` | Standard XANES descriptors extracted from each spectrum, if exported from Section 8. |
| RF regression summaries/predictions/feature importances | Generated by Section 11 from the RF manuscript experiment functions. |
| RF classification summaries/predictions/feature importances | Generated by Section 11 for B-rate and N-rate concentration classification. |
| baseline model summaries | Generated by Section 12 for RF, XGBoost, Lasso, and MLP under stratified and grouped split protocols. |

For the manuscript response, the key methodological clarification is that all intensity features are selected by named energy windows, not by positional column slicing.


## 14. XANES descriptor baseline

This section reproduces `ml_experiments/xanes_descriptor_baseline.py`, which compares \
**standard XANES descriptors** (peak positions, areas, edge/centroid features) against \
the **region-resolved intensity features** used in the manuscript.

### Descriptor groups

| Key | Name | Features |
|-----|------|---------|
| A\_peak | Peak-only | pi\* / sigma\* / post-edge: peak energy & height (6) |
| B\_area | Area-only | region areas, area ratios, height ratios (9) |
| C\_edge | Edge/centroid | centroid energies, edge derivative, half-max energy, post-edge slope/mean (9) |
| D\_all | All standard | union of A + B + C |
| E\_pi\_int | pi\* intensity | raw spectral intensities in pi\* window (265–273.5 eV) |
| E\_full\_int | Full intensity | raw spectral intensities in full window (265–290 eV) |

### Models
RF, XGBoost, Lasso (L1-regularised), MLP — evaluated with the same stratified 80/20 and \
grouped 5-fold CV protocols as Sections 11/12.

### Outputs (written to `xanes_descriptor_baseline/`)
- `xanes_descriptors.csv` — extracted descriptor values for all spectra
- `sample_splits.csv` — train/test/fold assignments
- `descriptor_regression_summary_stratified.csv` / `_grouped.csv`
- `descriptor_classification_summary_stratified.csv` / `_grouped.csv`
- `descriptor_comparison_tables.xlsx` — pivot tables (models × feature sets)
- `descriptor_vs_intensity_regression_bar.png` / `descriptor_vs_intensity_classification_bar.png`
- `conclusions.txt` — auto-generated comparison summary

> **Set `RUN_XANES_DESCRIPTOR_BASELINE = True` to execute this section.**
> Left `False` by default — runs ~72 model fits and takes several minutes.


In [36]:
# XANES descriptor baseline
# Adapted from ml_experiments/xanes_descriptor_baseline.py.
# Compares standard XANES descriptors (peak, area, edge/centroid features)
# against region-resolved intensity features (E_pi_int, E_full_int).
# Uses the same split protocol as Sections 10-12 (rf_results_v2 logic).

RUN_XANES_DESCRIPTOR_BASELINE = True  # set True to execute

if RUN_XANES_DESCRIPTOR_BASELINE:
    import warnings
    from scipy.stats import linregress
    from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
    from sklearn.linear_model import LassoCV, Lasso, LogisticRegression
    from sklearn.neural_network import MLPRegressor, MLPClassifier
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                                  accuracy_score, f1_score)
    from sklearn.model_selection import KFold, StratifiedKFold, StratifiedGroupKFold
    try:
        import xgboost as xgb
        _HAS_XGB = True
    except ImportError:
        _HAS_XGB = False
        print("xgboost not installed; XGB models will be skipped.")
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    warnings.filterwarnings("ignore")

    OUT_DESC = ROOT / "xanes_descriptor_baseline"
    OUT_DESC.mkdir(exist_ok=True)

    # ── Model hyperparameters ─────────────────────────────────────────
    _RF_REG  = dict(n_estimators=100, random_state=RANDOM_STATE)
    _RF_CLS  = dict(n_estimators=100, random_state=RANDOM_STATE)
    _XGB_REG = dict(n_estimators=300, max_depth=4, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8,
                    random_state=RANDOM_STATE, verbosity=0, n_jobs=1)
    _XGB_CLS = dict(n_estimators=200, max_depth=4, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8,
                    eval_metric="mlogloss", random_state=RANDOM_STATE,
                    verbosity=0, n_jobs=1)
    _MLP_REG = dict(hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
                    max_iter=3000, early_stopping=True, validation_fraction=0.15,
                    learning_rate_init=0.001, random_state=RANDOM_STATE)
    _MLP_CLS = dict(hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
                    max_iter=3000, early_stopping=True, validation_fraction=0.15,
                    random_state=RANDOM_STATE)

    N_SPLITS     = 5
    _CONC_BINS   = [0, 1.39, 2.78, 4.17, 5.56, 6.95, np.inf]
    _CONC_LABELS = [0.0, 1.39, 2.78, 4.17, 5.56, 6.95]

    # ════════════════════════════════════════════════════════════════
    # 1. DESCRIPTOR EXTRACTION
    # ════════════════════════════════════════════════════════════════

    def _win(e, s, lo, hi):
        mask = (e >= lo) & (e <= hi)
        return e[mask], s[mask]

    def _extract_descriptors(e, s):
        """Extract all standard XANES descriptors for a single spectrum."""
        d = {}
        _regions = {"pi": (265.0, 273.5), "sigma": (273.5, 283.5),
                    "postedge": (283.5, 290.0), "full": (265.0, 290.0)}
        areas = {}
        for reg, (lo, hi) in _regions.items():
            e_w, s_w = _win(e, s, lo, hi)
            if len(e_w) < 2:
                area = peak_e = peak_h = centroid = np.nan
            else:
                area   = np.trapz(s_w, e_w)
                pk     = np.argmax(s_w)
                peak_e = float(e_w[pk]); peak_h = float(s_w[pk])
                with np.errstate(invalid="ignore", divide="ignore"):
                    denom    = np.trapz(s_w, e_w)
                    centroid = (np.trapz(s_w * e_w, e_w) / denom
                                if denom != 0 else np.nan)
            d[f"{reg}_peak_energy"]  = peak_e
            d[f"{reg}_peak_height"]  = peak_h
            d[f"{reg}_area"]         = area
            d[f"{reg}_centroid"]     = centroid
            areas[reg]               = area

        e_pe, s_pe = _win(e, s, 283.5, 290.0)
        if len(e_pe) >= 2:
            d["postedge_mean_intensity"] = float(np.mean(s_pe))
            slope, *_ = linregress(e_pe, s_pe)
            d["postedge_slope"] = float(slope)
        else:
            d["postedge_mean_intensity"] = d["postedge_slope"] = np.nan

        e_f, s_f = _win(e, s, 265.0, 290.0)
        if len(e_f) >= 2:
            d["full_mean_intensity"] = float(np.mean(s_f))
            d["full_std_intensity"]  = float(np.std(s_f))
        else:
            d["full_mean_intensity"] = d["full_std_intensity"] = np.nan

        full_a = areas.get("full", np.nan)
        for reg in ("pi", "sigma", "postedge"):
            r = areas.get(reg, np.nan)
            d[f"area_ratio_{reg}_full"] = (
                r / full_a
                if (full_a and not np.isnan(full_a) and full_a != 0) else np.nan)

        pi_h = d.get("pi_peak_height", np.nan)
        sg_h = d.get("sigma_peak_height", np.nan)
        d["height_ratio_pi_sigma"] = (
            pi_h / sg_h if (sg_h and not np.isnan(sg_h) and sg_h != 0) else np.nan)
        if len(e_f) >= 2:
            fmh = float(np.max(s_f))
            d["height_ratio_pi_full"] = (
                pi_h / fmh if (fmh != 0 and not np.isnan(pi_h)) else np.nan)
        else:
            d["height_ratio_pi_full"] = np.nan

        e_pi, s_pi = _win(e, s, 265.0, 273.5)
        if len(e_pi) >= 3:
            grad = np.gradient(s_pi, e_pi)
            pk   = np.argmax(grad)
            d["edge_derivative_energy"] = float(e_pi[pk])
            d["edge_derivative_max"]    = float(grad[pk])
        else:
            d["edge_derivative_energy"] = d["edge_derivative_max"] = np.nan

        if len(e_f) >= 2:
            half = np.max(s_f) / 2.0
            idx_a = np.where(s_f >= half)[0]
            d["half_max_energy"] = float(e_f[idx_a[0]]) if len(idx_a) > 0 else np.nan
        else:
            d["half_max_energy"] = np.nan
        return d

    # Descriptor group definitions
    _FEAT_GROUPS = {
        "A_peak": ["pi_peak_energy", "pi_peak_height",
                   "sigma_peak_energy", "sigma_peak_height",
                   "postedge_peak_energy", "postedge_peak_height"],
        "B_area": ["pi_area", "sigma_area", "postedge_area", "full_area",
                   "area_ratio_pi_full", "area_ratio_sigma_full", "area_ratio_postedge_full",
                   "height_ratio_pi_sigma", "height_ratio_pi_full"],
        "C_edge": ["pi_centroid", "sigma_centroid", "postedge_centroid", "full_centroid",
                   "edge_derivative_energy", "edge_derivative_max", "half_max_energy",
                   "postedge_mean_intensity", "postedge_slope"],
    }
    _FEAT_GROUPS["D_all"] = sorted(
        set(c for cols in _FEAT_GROUPS.values() for c in cols))

    _GROUP_DISPLAY = {
        "A_peak":     "Peak-only descriptors",
        "B_area":     "Area-only descriptors",
        "C_edge":     "Edge/centroid descriptors",
        "D_all":      "All standard descriptors",
        "E_pi_int":   "pi* intensity features",
        "E_full_int": "Full-spectrum intensity features",
    }

    # ════════════════════════════════════════════════════════════════
    # 2. EXTRACT DESCRIPTORS FOR ALL SPECTRA
    # ════════════════════════════════════════════════════════════════
    print("Extracting descriptors for", len(spectra_names), "spectra...")
    _desc_records = []
    for _i, _name in enumerate(spectra_names):
        _d = _extract_descriptors(energies, spectra_matrix[_i])
        _d["spectrum_name"] = _name
        _desc_records.append(_d)
    desc_df = pd.DataFrame(_desc_records)

    for _col in ["configuration_id", "dopant_family", "concentration", "type",
                 "mbl", "bader", "stratify_key", "parsed_group"]:
        if _col in meta_split.columns:
            desc_df[_col] = meta_split[_col].values

    _nan_c = desc_df[_FEAT_GROUPS["D_all"]].isna().sum()
    _nan_c = _nan_c[_nan_c > 0]
    if len(_nan_c):
        print("  NaN descriptor counts:")
        for _col, _cnt in _nan_c.items():
            print(f"    {_col}: {_cnt} ({_cnt/len(desc_df)*100:.1f}%)")
    else:
        print("  No NaN descriptors.")

    _meta_cols = ["spectrum_name", "configuration_id", "dopant_family",
                  "concentration", "type", "mbl", "bader", "stratify_key"]
    _all_dcols = sorted(set(c for cols in _FEAT_GROUPS.values() for c in cols))
    _out_cols  = [c for c in _meta_cols if c in desc_df.columns] + _all_dcols
    desc_df[_out_cols].to_csv(OUT_DESC / "xanes_descriptors.csv", index=False)
    print(f"  Saved: xanes_descriptors.csv ({len(desc_df)} rows, {len(_all_dcols)} descriptor cols)")

    print("\n  Descriptor groups:")
    for _grp, _cols in _FEAT_GROUPS.items():
        _valid = [c for c in _cols if c in desc_df.columns]
        print(f"    {_grp} ({_GROUP_DISPLAY[_grp]}): {len(_valid)} features")

    # Save split assignments
    _split_df = pd.DataFrame({"spectrum": spectra_names, "split_mbl": "excluded"})
    _split_df.loc[train_idx, "split_mbl"] = "train"
    _split_df.loc[test_idx,  "split_mbl"] = "test"
    _split_df["grouped_fold"] = fold_assignment
    _split_df.to_csv(OUT_DESC / "sample_splits.csv", index=False)
    print(f"  Saved: sample_splits.csv (train={len(train_idx)}, test={len(test_idx)})")

    # ════════════════════════════════════════════════════════════════
    # 3. BUILD FEATURE MATRICES
    # ════════════════════════════════════════════════════════════════
    X_desc = {grp: desc_df[cols].values.astype(float)
              for grp, cols in _FEAT_GROUPS.items()}
    _pi_mask   = (energies >= 265.0) & (energies <= 273.5)
    _full_mask  = (energies >= 265.0) & (energies <= 290.0)
    X_desc["E_pi_int"]   = spectra_matrix[:, _pi_mask]
    X_desc["E_full_int"] = spectra_matrix[:, _full_mask]

    print("\n  Feature matrix sizes:")
    for _grp, _X in X_desc.items():
        print(f"    {_grp} ({_GROUP_DISPLAY.get(_grp, _grp)}): {_X.shape[1]} features")

    # ════════════════════════════════════════════════════════════════
    # 4. MODEL HELPERS
    # ════════════════════════════════════════════════════════════════

    def _impute(X_tr, X_te):
        X_tr = np.asarray(X_tr, dtype=float); X_te = np.asarray(X_te, dtype=float)
        cm = np.nanmean(X_tr, axis=0)
        cm = np.where(np.isnan(cm), 0.0, cm)
        return (np.where(np.isnan(X_tr), cm[None, :], X_tr),
                np.where(np.isnan(X_te), cm[None, :], X_te))

    def _reg_m(yt, yp):
        return (float(r2_score(yt, yp)),
                float(np.sqrt(mean_squared_error(yt, yp))),
                float(mean_absolute_error(yt, yp)))

    def _cls_m(yt, yp):
        return (float(accuracy_score(yt, yp)),
                float(f1_score(yt, yp, average="macro", zero_division=0)))

    def _fit_reg(mn, X_tr, y_tr, X_te):
        if mn == "Lasso":
            xs = StandardScaler(); ys = StandardScaler()
            Xt = xs.fit_transform(X_tr); Xv = xs.transform(X_te)
            yt = ys.fit_transform(y_tr.reshape(-1, 1)).ravel()
            lcv = LassoCV(cv=5, max_iter=50000, random_state=RANDOM_STATE); lcv.fit(Xt, yt)
            m = Lasso(alpha=lcv.alpha_, max_iter=50000); m.fit(Xt, yt)
            return ys.inverse_transform(m.predict(Xv).reshape(-1, 1)).ravel()
        if mn == "MLP":
            xs = StandardScaler(); ys = StandardScaler()
            Xt = xs.fit_transform(X_tr); Xv = xs.transform(X_te)
            yt = ys.fit_transform(y_tr.reshape(-1, 1)).ravel()
            m = MLPRegressor(**_MLP_REG); m.fit(Xt, yt)
            return ys.inverse_transform(m.predict(Xv).reshape(-1, 1)).ravel()
        if mn == "XGB" and _HAS_XGB:
            m = xgb.XGBRegressor(**_XGB_REG); m.fit(X_tr, y_tr); return m.predict(X_te)
        m = RandomForestRegressor(**_RF_REG); m.fit(X_tr, y_tr); return m.predict(X_te)

    def _cv_reg(mn, X_tr, y_tr, n=5):
        kf = KFold(n_splits=n, shuffle=True, random_state=RANDOM_STATE)
        sc = [r2_score(y_tr[va], _fit_reg(mn, X_tr[tr], y_tr[tr], X_tr[va]))
              for tr, va in kf.split(X_tr)]
        return float(np.mean(sc)), float(np.std(sc))

    def _fit_cls(mn, X_tr, y_tr, X_te):
        y_tr = np.asarray(y_tr).ravel()
        cls  = sorted(set(y_tr.tolist()))
        c2i  = {c: i for i, c in enumerate(cls)}
        i2c  = {i: c for c, i in c2i.items()}
        ye   = np.array([c2i[c] for c in y_tr], dtype=int)
        if mn == "Lasso":
            xs = StandardScaler()
            Xt = xs.fit_transform(np.asarray(X_tr, dtype=float))
            Xv = xs.transform(np.asarray(X_te, dtype=float))
            m  = LogisticRegression(penalty="l1", solver="saga", C=1.0,
                                    max_iter=5000, random_state=RANDOM_STATE)
            m.fit(Xt, ye); p = m.predict(Xv)
        elif mn == "MLP":
            xs = StandardScaler()
            Xt = xs.fit_transform(np.asarray(X_tr, dtype=float))
            Xv = xs.transform(np.asarray(X_te, dtype=float))
            m  = MLPClassifier(**_MLP_CLS); m.fit(Xt, ye); p = m.predict(Xv)
        elif mn == "XGB" and _HAS_XGB:
            m = xgb.XGBClassifier(**_XGB_CLS)
            m.fit(np.asarray(X_tr, dtype=float), ye)
            p = m.predict(np.asarray(X_te, dtype=float))
        else:
            m = RandomForestClassifier(**_RF_CLS)
            m.fit(np.asarray(X_tr, dtype=float), ye)
            p = m.predict(np.asarray(X_te, dtype=float))
        return np.array([i2c[int(x)] for x in p])

    def _cv_cls(mn, X_tr, y_tr, n=5):
        skf = StratifiedKFold(n_splits=n, shuffle=True, random_state=RANDOM_STATE)
        acc, f1 = [], []
        for tr, va in skf.split(X_tr, y_tr):
            yp = _fit_cls(mn, X_tr[tr], y_tr[tr], X_tr[va])
            acc.append(accuracy_score(y_tr[va], yp))
            f1.append(f1_score(y_tr[va], yp, average="macro", zero_division=0))
        return float(np.mean(acc)), float(np.std(acc)), float(np.mean(f1)), float(np.std(f1))

    # ════════════════════════════════════════════════════════════════
    # 5. REGRESSION EXPERIMENTS
    # ════════════════════════════════════════════════════════════════
    print("\n" + "="*70)
    print("REGRESSION EXPERIMENTS")
    print("="*70)

    _MODELS_REG  = (["RF", "XGB", "Lasso", "MLP"] if _HAS_XGB
                    else ["RF", "Lasso", "MLP"])
    _REG_TARGETS = {"mbl": "Mean Bond Length", "bader": "Mean Bader Charge"}

    _reg_strat, _reg_grp = [], []

    for _tgt in _REG_TARGETS:
        _y = meta_split[_tgt].values.astype(float)
        _tr, _te = spectrum_level_split(meta_split, target_col=_tgt)
        _fa = grouped_cv_folds(meta_split, target_col=_tgt)
        _vi = np.where(_fa >= 0)[0]; _fv = _fa[_vi]

        for _mn in _MODELS_REG:
            for _grp, _Xm in X_desc.items():
                # Stratified 80/20
                _Xt, _Xv = _impute(_Xm[_tr], _Xm[_te])
                _yp = _fit_reg(_mn, _Xt, _y[_tr], _Xv)
                _r2, _rmse, _mae = _reg_m(_y[_te], _yp)
                _cvr2, _cvs = _cv_reg(_mn, _Xt, _y[_tr])
                print(f"  REG strat [{_tgt}][{_grp}][{_mn}] "
                      f"CV R2={_cvr2:.4f}  Test R2={_r2:.4f}  MAE={_mae:.4f}")
                _reg_strat.append({
                    "Target": _tgt, "Feature_set": _grp,
                    "Feature_set_display": _GROUP_DISPLAY.get(_grp, _grp),
                    "Model": _mn, "N_features": _Xt.shape[1],
                    "CV_R2_mean": _cvr2, "CV_R2_std": _cvs,
                    "Test_R2": _r2, "Test_RMSE": _rmse, "Test_MAE": _mae,
                })

                # Grouped 5-fold OOF
                _ot = np.full(len(_vi), np.nan); _op = np.full(len(_vi), np.nan)
                _fcr2 = []
                for _fid in range(N_SPLITS):
                    _vm = _fv == _fid; _tm = (_fv != _fid) & (_fv != -1)
                    _vix = _vi[_vm]; _tix = _vi[_tm]
                    _Xg, _Xgv = _impute(_Xm[_tix], _Xm[_vix])
                    _ypg = _fit_reg(_mn, _Xg, _y[_tix], _Xgv)
                    _fcr2.append(r2_score(_y[_vix], _ypg))
                    _ot[_vm] = _y[_vix]; _op[_vm] = _ypg
                _ok = ~np.isnan(_ot)
                _or2, _ormse, _omae = _reg_m(_ot[_ok], _op[_ok])
                _fcr2a = np.array(_fcr2)
                print(f"  REG grp   [{_tgt}][{_grp}][{_mn}] "
                      f"OOF R2={_or2:.4f}  OOF MAE={_omae:.4f}")
                _reg_grp.append({
                    "Target": _tgt, "Feature_set": _grp,
                    "Feature_set_display": _GROUP_DISPLAY.get(_grp, _grp),
                    "Model": _mn, "N_features": _Xt.shape[1],
                    "CV_R2_mean": float(_fcr2a.mean()), "CV_R2_std": float(_fcr2a.std()),
                    "OOF_R2": _or2, "OOF_RMSE": _ormse, "OOF_MAE": _omae,
                })

    df_reg_strat_d   = pd.DataFrame(_reg_strat)
    df_reg_grouped_d = pd.DataFrame(_reg_grp)
    df_reg_strat_d.to_csv(OUT_DESC / "descriptor_regression_summary_stratified.csv",   index=False)
    df_reg_grouped_d.to_csv(OUT_DESC / "descriptor_regression_summary_grouped.csv", index=False)
    print("\n  Saved: descriptor_regression_summary_stratified.csv")
    print("  Saved: descriptor_regression_summary_grouped.csv")

    # ════════════════════════════════════════════════════════════════
    # 6. CLASSIFICATION EXPERIMENTS
    # ════════════════════════════════════════════════════════════════
    print("\n" + "="*70)
    print("CLASSIFICATION EXPERIMENTS")
    print("="*70)

    _MODELS_CLS = (["RF", "XGB", "Lasso", "MLP"] if _HAS_XGB
                   else ["RF", "Lasso", "MLP"])
    _cls_strat, _cls_grp = [], []

    for _dop in ["B", "N"]:
        _cc  = f"{_dop}_rate"
        _cv  = meta_split[_cc].values.astype(float)
        _ycls = np.array([
            _CONC_LABELS[np.searchsorted(_CONC_BINS[1:], c, side="right")]
            if not np.isnan(c) else np.nan for c in _cv
        ], dtype=object)

        _pg  = meta_split["parsed_group"].values
        _vci = np.where([(_g == _dop or _g == "pristine") for _g in _pg])[0]
        _ycv = np.array([str(v) for v in _ycls[_vci]])
        _sk  = meta_split["stratify_key"].values[_vci]

        _trl, _tel = train_test_split(
            np.arange(len(_vci)), test_size=0.2,
            random_state=RANDOM_STATE, stratify=_sk)
        _trc = _vci[_trl]; _tec = _vci[_tel]
        _ytrc = _ycv[_trl]; _ytec = _ycv[_tel]

        _cids = meta_split["configuration_id"].values[_vci]
        _sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
        _fcls = np.full(len(_vci), -1, dtype=int)
        for _fid, (_, _vl) in enumerate(_sgkf.split(np.zeros(len(_vci)), _sk, _cids)):
            _fcls[_vl] = _fid

        for _mn in _MODELS_CLS:
            for _grp, _Xm in X_desc.items():
                # Stratified 80/20
                _Xt, _Xv = _impute(_Xm[_trc], _Xm[_tec])
                _ytp = _fit_cls(_mn, _Xt, _ytrc, _Xv)
                _ta, _tf = _cls_m(_ytec, _ytp)
                _ca, _cas, _cf, _cfs = _cv_cls(_mn, _Xt, _ytrc)
                print(f"  CLS strat [{_dop}][{_grp}][{_mn}] "
                      f"CV Acc={_ca:.4f}  Test Acc={_ta:.4f}  F1={_tf:.4f}")
                _cls_strat.append({
                    "Dopant": _dop, "Feature_set": _grp,
                    "Feature_set_display": _GROUP_DISPLAY.get(_grp, _grp),
                    "Model": _mn, "N_features": _Xt.shape[1],
                    "CV_Acc_mean": _ca, "CV_Acc_std": _cas,
                    "CV_F1_mean": _cf, "CV_F1_std": _cfs,
                    "Test_Acc": _ta, "Test_F1": _tf,
                })

                # Grouped 5-fold OOF
                _otr, _opr, _fal = [], [], []
                for _fid in range(N_SPLITS):
                    _vm = _fcls == _fid; _tm = (_fcls != _fid) & (_fcls != -1)
                    if _vm.sum() == 0: continue
                    _vi = _vci[_vm]; _ti = _vci[_tm]
                    _Xg, _Xgv = _impute(_Xm[_ti], _Xm[_vi])
                    _ypg = _fit_cls(_mn, _Xg, _ycv[_tm], _Xgv)
                    _otr.extend(_ycv[_vm]); _opr.extend(_ypg)
                    _fal.append(accuracy_score(_ycv[_vm], _ypg))
                _oa, _of = _cls_m(np.array(_otr), np.array(_opr))
                print(f"  CLS grp   [{_dop}][{_grp}][{_mn}] "
                      f"OOF Acc={_oa:.4f}  OOF F1={_of:.4f}")
                _cls_grp.append({
                    "Dopant": _dop, "Feature_set": _grp,
                    "Feature_set_display": _GROUP_DISPLAY.get(_grp, _grp),
                    "Model": _mn, "N_features": _Xt.shape[1],
                    "CV_Acc_mean": float(np.mean(_fal)),
                    "OOF_Acc": _oa, "OOF_F1": _of,
                })

    df_cls_strat_d   = pd.DataFrame(_cls_strat)
    df_cls_grouped_d = pd.DataFrame(_cls_grp)
    df_cls_strat_d.to_csv(OUT_DESC / "descriptor_classification_summary_stratified.csv",  index=False)
    df_cls_grouped_d.to_csv(OUT_DESC / "descriptor_classification_summary_grouped.csv", index=False)
    print("\n  Saved: descriptor_classification_summary_stratified.csv")
    print("  Saved: descriptor_classification_summary_grouped.csv")

    # ════════════════════════════════════════════════════════════════
    # 7. EXCEL COMPARISON TABLE
    # ════════════════════════════════════════════════════════════════
    with pd.ExcelWriter(OUT_DESC / "descriptor_comparison_tables.xlsx",
                        engine="openpyxl") as _w:
        for _tgt in ("mbl", "bader"):
            for _sl, _df in [("Stratified", df_reg_strat_d), ("Grouped", df_reg_grouped_d)]:
                _r2c  = "Test_R2"  if _sl == "Stratified" else "OOF_R2"
                _maec = "Test_MAE" if _sl == "Stratified" else "OOF_MAE"
                _rmsec = "Test_RMSE" if _sl == "Stratified" else "OOF_RMSE"
                _rows = []
                for _, _row in _df[_df["Target"] == _tgt].iterrows():
                    _rows.append({
                        "Feature representation": _row["Feature_set_display"],
                        "Feature set key": _row["Feature_set"],
                        "Model": _row["Model"], "Split": _sl,
                        "N features": int(_row["N_features"]),
                        "CV R2 mean": round(_row["CV_R2_mean"], 4),
                        "CV R2 std":  round(_row["CV_R2_std"],  4),
                        "R2":   round(_row[_r2c],   4),
                        "RMSE": round(_row.get(_rmsec, np.nan), 6),
                        "MAE":  round(_row[_maec],  6),
                    })
                pd.DataFrame(_rows).to_excel(
                    _w, sheet_name=f"Reg_{_tgt[:3]}_{_sl[:5]}", index=False)
        for _dop in ("B", "N"):
            for _sl, _df in [("Stratified", df_cls_strat_d), ("Grouped", df_cls_grouped_d)]:
                _acc = "Test_Acc" if _sl == "Stratified" else "OOF_Acc"
                _f1  = "Test_F1"  if _sl == "Stratified" else "OOF_F1"
                _rows = []
                for _, _row in _df[_df["Dopant"] == _dop].iterrows():
                    _rows.append({
                        "Feature representation": _row["Feature_set_display"],
                        "Feature set key": _row["Feature_set"],
                        "Model": _row["Model"], "Split": _sl,
                        "N features": int(_row["N_features"]),
                        "CV Acc mean": round(_row["CV_Acc_mean"], 4),
                        "Accuracy": round(_row[_acc], 4),
                        "F1":       round(_row[_f1],  4),
                    })
                pd.DataFrame(_rows).to_excel(
                    _w, sheet_name=f"Cls_{_dop}_{_sl[:5]}", index=False)
    print("\n  Saved: descriptor_comparison_tables.xlsx")

    # ════════════════════════════════════════════════════════════════
    # 8. COMPARISON PLOTS
    # ════════════════════════════════════════════════════════════════
    _PG  = ["D_all", "E_pi_int", "E_full_int"]
    _PL  = {"D_all": "All descriptors", "E_pi_int": "pi* intensity",
            "E_full_int": "Full intensity"}
    _PC  = {"D_all": "#4472C4", "E_pi_int": "#ED7D31", "E_full_int": "#70AD47"}

    _fig, _axes = plt.subplots(2, 2, figsize=(14, 10))
    for _ri, (_tgt, _ms, _mg, _yl) in enumerate([
        ("mbl",   "Test_MAE", "OOF_MAE", "MAE (Ang.)"),
        ("bader", "Test_MAE", "OOF_MAE", "MAE (e)"),
    ]):
        for _ci, (_sl, _df, _mc) in enumerate([
            ("Stratified",  df_reg_strat_d,   _ms),
            ("Grouped OOF", df_reg_grouped_d, _mg),
        ]):
            _ax = _axes[_ri][_ci]
            _vals = []
            for _g in _PG:
                _r = _df[(_df["Target"] == _tgt) & (_df["Feature_set"] == _g) &
                         (_df["Model"] == "RF")]
                _vals.append(float(_r[_mc].values[0]) if len(_r) else 0.0)
            _bars = _ax.bar(np.arange(len(_PG)), _vals, width=0.6,
                            color=[_PC[g] for g in _PG],
                            edgecolor="black", linewidth=0.7, alpha=0.85)
            _ax.set_xticks(np.arange(len(_PG)))
            _ax.set_xticklabels([_PL[g] for g in _PG], fontsize=10)
            _ax.set_ylabel(_yl, fontsize=12)
            _ax.set_title(f"RF | {_tgt.upper()} | {_sl}", fontsize=11)
            for _b, _v in zip(_bars, _vals):
                _ax.text(_b.get_x() + _b.get_width()/2, _v + max(_vals)*0.01,
                         f"{_v:.3f}", ha="center", va="bottom", fontsize=9)
    _fig.tight_layout()
    _fig.savefig(OUT_DESC / "descriptor_vs_intensity_regression_bar.png", dpi=600)
    plt.close(_fig)
    print("  Saved: descriptor_vs_intensity_regression_bar.png")

    _fig2, _axes2 = plt.subplots(2, 2, figsize=(14, 10))
    for _ri, _dop in enumerate(["B", "N"]):
        for _ci, (_sl, _df, _fc) in enumerate([
            ("Stratified",  df_cls_strat_d,   "Test_F1"),
            ("Grouped OOF", df_cls_grouped_d, "OOF_F1"),
        ]):
            _ax = _axes2[_ri][_ci]
            _vals = []
            for _g in _PG:
                _r = _df[(_df["Dopant"] == _dop) & (_df["Feature_set"] == _g) &
                         (_df["Model"] == "RF")]
                _vals.append(float(_r[_fc].values[0]) if len(_r) else 0.0)
            _bars = _ax.bar(np.arange(len(_PG)), _vals, width=0.6,
                            color=[_PC[g] for g in _PG],
                            edgecolor="black", linewidth=0.7, alpha=0.85)
            _ax.set_xticks(np.arange(len(_PG)))
            _ax.set_xticklabels([_PL[g] for g in _PG], fontsize=10)
            _ax.set_ylabel("F1 (macro)", fontsize=12)
            _ax.set_title(f"RF | {_dop}-doped conc. | {_sl}", fontsize=11)
            _ax.set_ylim(0, 1.05)
            for _b, _v in zip(_bars, _vals):
                _ax.text(_b.get_x() + _b.get_width()/2, _v + 0.01,
                         f"{_v:.3f}", ha="center", va="bottom", fontsize=9)
    _fig2.tight_layout()
    _fig2.savefig(OUT_DESC / "descriptor_vs_intensity_classification_bar.png", dpi=600)
    plt.close(_fig2)
    print("  Saved: descriptor_vs_intensity_classification_bar.png")

    # ════════════════════════════════════════════════════════════════
    # 9. SUMMARY CONCLUSIONS
    # ════════════════════════════════════════════════════════════════
    print("\n" + "="*70)
    print("CONCLUSIONS (RF, stratified test set)")
    print("="*70)
    _desc_grps = ["A_peak", "B_area", "C_edge", "D_all"]
    _concl = []

    for _tgt in ("mbl", "bader"):
        _unit = "Ang." if _tgt == "mbl" else "e"
        _sub  = df_reg_strat_d[(df_reg_strat_d["Target"] == _tgt) &
                                (df_reg_strat_d["Model"] == "RF") &
                                (df_reg_strat_d["Feature_set"].isin(_desc_grps))]
        if len(_sub) == 0: continue
        _best = _sub.loc[_sub["Test_MAE"].idxmin()]
        _pi_r = df_reg_strat_d[(df_reg_strat_d["Target"] == _tgt) &
                                (df_reg_strat_d["Feature_set"] == "E_pi_int") &
                                (df_reg_strat_d["Model"] == "RF")]
        _fl_r = df_reg_strat_d[(df_reg_strat_d["Target"] == _tgt) &
                                (df_reg_strat_d["Feature_set"] == "E_full_int") &
                                (df_reg_strat_d["Model"] == "RF")]
        _pi_mae = float(_pi_r["Test_MAE"].values[0]) if len(_pi_r) else np.nan
        _fl_mae = float(_fl_r["Test_MAE"].values[0]) if len(_fl_r) else np.nan
        print(f"\n{_tgt.upper()} | best descriptor: {_best['Feature_set']} "
              f"MAE={float(_best['Test_MAE']):.4f} {_unit}  "
              f"pi*-int MAE={_pi_mae:.4f}  full-int MAE={_fl_mae:.4f}")
        if not np.isnan(_fl_mae) and _fl_mae != 0:
            _gap = (float(_best["Test_MAE"]) - _fl_mae) / _fl_mae * 100
            _dir = "outperforms" if _gap > 0 else "matches"
            _concl.append(f"{_tgt.upper()}: intensity features {_dir} best descriptor "
                          f"baseline by {abs(_gap):.1f}% MAE.")

    for _dop in ("B", "N"):
        _sub  = df_cls_strat_d[(df_cls_strat_d["Dopant"] == _dop) &
                                (df_cls_strat_d["Model"] == "RF") &
                                (df_cls_strat_d["Feature_set"].isin(_desc_grps))]
        if len(_sub) == 0: continue
        _best = _sub.loc[_sub["Test_F1"].idxmax()]
        _pi_c = df_cls_strat_d[(df_cls_strat_d["Dopant"] == _dop) &
                                (df_cls_strat_d["Feature_set"] == "E_pi_int") &
                                (df_cls_strat_d["Model"] == "RF")]
        _fl_c = df_cls_strat_d[(df_cls_strat_d["Dopant"] == _dop) &
                                (df_cls_strat_d["Feature_set"] == "E_full_int") &
                                (df_cls_strat_d["Model"] == "RF")]
        _pi_f = float(_pi_c["Test_F1"].values[0]) if len(_pi_c) else np.nan
        _fl_f = float(_fl_c["Test_F1"].values[0]) if len(_fl_c) else np.nan
        print(f"{_dop}-conc | best descriptor: {_best['Feature_set']} "
              f"F1={float(_best['Test_F1']):.4f}  "
              f"pi*-int F1={_pi_f:.4f}  full-int F1={_fl_f:.4f}")
        if not np.isnan(_fl_f) and _fl_f != 0:
            _gap = (_fl_f - float(_best["Test_F1"])) / _fl_f * 100
            _dir = "outperforms" if _gap > 0 else "matches"
            _concl.append(f"{_dop}-concentration: intensity features {_dir} best "
                          f"descriptor baseline by {abs(_gap):.1f}% F1.")

    with open(OUT_DESC / "conclusions.txt", "w", encoding="utf-8") as _f:
        _f.write("XANES Descriptor Baseline vs Region-Resolved Intensity Features\n")
        _f.write("=" * 70 + "\n\n")
        for _l in _concl:
            _f.write(_l + "\n\n")
    print(f"\n  Saved: conclusions.txt")
    print(f"\nAll outputs written to: {OUT_DESC}")
    print("=" * 70)


Extracting descriptors for 415 spectra...
  No NaN descriptors.
  Saved: xanes_descriptors.csv (415 rows, 24 descriptor cols)

  Descriptor groups:
    A_peak (Peak-only descriptors): 6 features
    B_area (Area-only descriptors): 9 features
    C_edge (Edge/centroid descriptors): 9 features
    D_all (All standard descriptors): 24 features
  Saved: sample_splits.csv (train=332, test=83)

  Feature matrix sizes:
    A_peak (Peak-only descriptors): 6 features
    B_area (Area-only descriptors): 9 features
    C_edge (Edge/centroid descriptors): 9 features
    D_all (All standard descriptors): 24 features
    E_pi_int (pi* intensity features): 86 features
    E_full_int (Full-spectrum intensity features): 251 features

REGRESSION EXPERIMENTS
  REG strat [mbl][A_peak][RF] CV R2=0.9447  Test R2=0.9517  MAE=0.0049
  REG grp   [mbl][A_peak][RF] OOF R2=0.9226  OOF MAE=0.0053
  REG strat [mbl][B_area][RF] CV R2=0.9550  Test R2=0.9353  MAE=0.0041
  REG grp   [mbl][B_area][RF] OOF R2=0.9432  OOF

## 15. Software environment

Print the Python version and the versions of all libraries used in this notebook.


In [38]:
import sys, importlib

print(f"Python: {sys.version}")
print()

_LIBS = [
    "numpy", "pandas", "scipy",
    "sklearn",       # scikit-learn
    "matplotlib",         
    "xgboost",
    "openpyxl",
]

for _lib in _LIBS:
    try:
        _mod = importlib.import_module(_lib)
        _ver = getattr(_mod, "__version__", "(no __version__)")
    except ImportError:
        _ver = "NOT INSTALLED"
    _display = "scikit-learn" if _lib == "sklearn" else _lib
    print(f"{_display}: {_ver}")


Python: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]

numpy: 1.26.4
pandas: 2.2.2
scipy: 1.13.1
scikit-learn: 1.5.1
matplotlib: 3.9.2
xgboost: 3.3.0
openpyxl: 3.1.5
